# Unit 6. Shrinkage, Nearest Neighbours and High Dimensions

精算統計模型 (SOA Exam SRM/PA). The text of [unit06.pdf](https://github.com/chang-ye-tu/asm/blob/master/note/unit06.pdf) with every R chunk as a code cell.

<a href="https://colab.research.google.com/github/chang-ye-tu/asm/blob/master/colab/unit06.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

The notebook runs on the R runtime (on Colab: Runtime › Change runtime type › R). Run the setup cell first: it installs the packages and downloads the data this unit uses.

In [ ]:
# Setup: run this cell first. It installs the packages this unit uses and downloads
# its data files into data/, the paths the code below reads.
local({
  if (is.null(getOption("repos")) || identical(unname(getOption("repos")["CRAN"]), "@CRAN@"))
    options(repos = c(CRAN = "https://cloud.r-project.org"))
  os <- tryCatch(readLines("/etc/os-release"), error = function(e) character())
  code <- sub("^VERSION_CODENAME=", "", grep("^VERSION_CODENAME=", os, value = TRUE))
  if (length(code) == 1 && nzchar(code))  # Linux, e.g. Colab: prebuilt binary packages
    options(repos = c(CRAN = sprintf("https://packagemanager.posit.co/cran/__linux__/%s/latest", code)),
            HTTPUserAgent = sprintf("R/%s R (%s)", getRversion(), paste(getRversion(),
              R.version$platform, R.version$arch, R.version$os)))
})
pkgs <- c("FNN", "glmnet", "ISLR2", "leaps", "MASS", "pls")
new <- setdiff(pkgs, rownames(installed.packages()))
if (length(new)) install.packages(new)
options(width = 62, digits = 4, scipen = 4, repr.plot.width = 6.4, repr.plot.height = 3.0)

In [ ]:
library(ISLR2); library(glmnet); library(FNN)

## 1 Ridge Regression

> **Reading.** ISLR §6.2.1. SRM learning outcome 2(g).

**Remark** (The classical assumptions; Unit 2). A1–A5 are Unit 2's classical assumptions on $\mathbf{Y}=\mathbf{X}\boldsymbol{\beta}+\boldsymbol{\varepsilon}$ with $\mathbf{X}$ fixed: linearity (A1), zero-mean errors (A2), $\operatorname{var}(\boldsymbol{\varepsilon})=\sigma^2\mathbf{I}_n$ with $\sigma^2>0$ (A3), $\operatorname{rank}(\mathbf{X})=k+1$, with $n>k+1$ for inference (A4), and normal errors (A5).

### 1.1 The estimator

**Definition 6.1** (Ridge regression; ISLR §6.2.1). Let $\mathbf{Y}\in\mathbb{R}^n$ be the response and $\mathbf{X}$ the $n\times p$ matrix of the predictors, without a column of ones; no rank condition is imposed, and $p$ may exceed $n$. For $\lambda\geqslant0$ the **ridge regression** estimates $(\hat\beta_0,\hat{\boldsymbol{\beta}}_\lambda)$ minimise

$$
\sum_{i=1}^n\Bigl(y_i-\beta_0-\sum_{j=1}^px_{ij}\beta_j\Bigr)^2+\lambda\sum_{j=1}^p\beta_j^2
=\mathrm{RSS}+\lambda\|\boldsymbol{\beta}\|^2 ,
$$

$\lambda$ being the **tuning parameter** and $\lambda\|\boldsymbol{\beta}\|^2$ the **shrinkage penalty**, which leaves the intercept unpenalised. The predictors are usually **standardised** first, each being non-constant,

$$
\tilde x_{ij}=\frac{x_{ij}-\bar x_j}{\hat s_j},\qquad
\hat s_j^2=\frac1n\sum_{i=1}^n(x_{ij}-\bar x_j)^2 ,
$$

so that each column has mean $0$ and $\sum_i\tilde x_{ij}^2=n$; ridge coefficients of standardised predictors are the **standardised coefficients**.

**Proposition 6.2** (The intercept; ISLR §6.2.1). If the columns of $\mathbf{X}$ are centred, $\mathbf{1}'\mathbf{X}=\mathbf{0}'$, the ridge estimates are the pairs $(\bar y,\hat{\boldsymbol{\beta}}_\lambda)$ with $\hat{\boldsymbol{\beta}}_\lambda$ a minimiser of $Q_\lambda(\boldsymbol{\beta})=\|\mathbf{Y}_c-\mathbf{X}\boldsymbol{\beta}\|^2+\lambda\|\boldsymbol{\beta}\|^2$, $\mathbf{Y}_c=\mathbf{Y}-\bar y\mathbf{1}$.

**Proof.** For fixed $\boldsymbol{\beta}$ the criterion is $\|(\mathbf{Y}-\mathbf{X}\boldsymbol{\beta})-\beta_0\mathbf{1}\|^2+\lambda\|\boldsymbol{\beta}\|^2$, a quadratic in $\beta_0$ with leading coefficient $n$, uniquely minimised at $\beta_0=\mathbf{1}'(\mathbf{Y}-\mathbf{X}\boldsymbol{\beta})/n=\bar y$ because $\mathbf{1}'\mathbf{X}=\mathbf{0}'$, with minimum $Q_\lambda(\boldsymbol{\beta})$. So $(\beta_0,\boldsymbol{\beta})$ is a minimiser iff $\beta_0=\bar y$ and $\boldsymbol{\beta}$ minimises $Q_\lambda$. ∎

**Theorem 6.3** (Existence, uniqueness and closed form; ISLR §6.2.1). For every $\lambda>0$, $\mathbf{X}'\mathbf{X}+\lambda\mathbf{I}_p$ is positive definite and $Q_\lambda$ has exactly one minimiser,

$$
\hat{\boldsymbol{\beta}}_\lambda=(\mathbf{X}'\mathbf{X}+\lambda\mathbf{I}_p)^{-1}\mathbf{X}'\mathbf{Y}_c ,
$$

whatever the rank of $\mathbf{X}$.

**Proof.** For $\mathbf{c}\neq\mathbf{0}$, $\mathbf{c}'(\mathbf{X}'\mathbf{X}+\lambda\mathbf{I})\mathbf{c}=\|\mathbf{X}\mathbf{c}\|^2+\lambda\|\mathbf{c}\|^2>0$, so $\mathbf{A}=\mathbf{X}'\mathbf{X}+\lambda\mathbf{I}$ is positive definite, hence invertible by the prerequisites' proposition on positive definite matrices. Put $\hat{\boldsymbol{\beta}}=\mathbf{A}^{-1}\mathbf{X}'\mathbf{Y}_c$. Expanding, $Q_\lambda(\boldsymbol{\beta})=\|\mathbf{Y}_c\|^2-2\boldsymbol{\beta}'\mathbf{X}'\mathbf{Y}_c+\boldsymbol{\beta}'\mathbf{A}\boldsymbol{\beta}$, and since $\mathbf{X}'\mathbf{Y}_c=\mathbf{A}\hat{\boldsymbol{\beta}}$,

$$
Q_\lambda(\boldsymbol{\beta})=\|\mathbf{Y}_c\|^2-\hat{\boldsymbol{\beta}}'\mathbf{A}\hat{\boldsymbol{\beta}}+(\boldsymbol{\beta}-\hat{\boldsymbol{\beta}})'\mathbf{A}(\boldsymbol{\beta}-\hat{\boldsymbol{\beta}}) ,
$$

whose last term is positive unless $\boldsymbol{\beta}=\hat{\boldsymbol{\beta}}$. ∎

**Remark** (ISLR §6.2.1). At $\lambda=0$ the criterion is least squares, whose minimiser is unique only if $\operatorname{rank}(\mathbf{X})=p$ (Unit 2's theorem **the least squares estimator** and remark **without full rank**); for $\lambda>0$ ridge has one estimate even when $p>n$.

**Proposition 6.4** (Scale equivariance fails for ridge; ISLR §6.2.1). Let the columns of $\mathbf{X}$ be centred, $\mathbf{D}=\operatorname{diag}(c_1,\ldots,c_p)$ with every $c_j>0$, and replace $\mathbf{X}$ by $\mathbf{X}\mathbf{D}$, the $j$th predictor multiplied by $c_j$.

(i) If $\operatorname{rank}(\mathbf{X})=p$, the least squares coefficients become $\mathbf{D}^{-1}\hat{\boldsymbol{\beta}}$ and the fitted values are unchanged.

(ii) The ridge coefficients become $\mathbf{D}^{-1}\hat{\boldsymbol{\gamma}}$, where $\hat{\boldsymbol{\gamma}}$ minimises $\|\mathbf{Y}_c-\mathbf{X}\boldsymbol{\gamma}\|^2+\lambda\sum_j\gamma_j^2/c_j^2$: a rescaled predictor changes the penalty on every coordinate it carries weight on, and with it the whole fit.

**Proof.** Put $\boldsymbol{\gamma}=\mathbf{D}\boldsymbol{\beta}$, so that $\mathbf{X}\mathbf{D}\boldsymbol{\beta}=\mathbf{X}\boldsymbol{\gamma}$. The sum of squares depends on $\boldsymbol{\beta}$ only through $\boldsymbol{\gamma}$, which gives (i) (Unit 3's proposition **codings are reparametrisations** with $\mathbf{A}=\operatorname{diag}(1,\mathbf{D})$), and $\|\boldsymbol{\beta}\|^2=\sum_j\gamma_j^2/c_j^2$, which gives (ii). ∎

**Remark** (ISLR §6.2.1). A change of units carries no information about the response but changes the ridge fit, so the predictors are standardised by Definition 6.1; the fit then does not depend on their units. `glmnet` standardises with the same divisor $n$.

### 1.2 Shrinkage along the principal components

**Theorem 6.5** (Ridge through the singular value decomposition; ISLR §6.2.1). Let the columns of $\mathbf{X}$ be centred and let $\mathbf{X}=\mathbf{U}\mathbf{D}\mathbf{V}'$ be its singular value decomposition (the prerequisites' theorem **SVD**), $r=\operatorname{rank}(\mathbf{X})$, $\mathbf{U}=[\mathbf{u}_1\ \cdots\ \mathbf{u}_r]$, $\mathbf{V}=[\mathbf{v}_1\ \cdots\ \mathbf{v}_r]$, $d_1\geqslant\cdots\geqslant d_r>0$. For $\lambda>0$,

$$
\hat{\boldsymbol{\beta}}_\lambda=\sum_{j=1}^r\frac{d_j}{d_j^2+\lambda}(\mathbf{u}_j'\mathbf{Y})\mathbf{v}_j ,\qquad
\mathbf{X}\hat{\boldsymbol{\beta}}_\lambda=\sum_{j=1}^r\frac{d_j^2}{d_j^2+\lambda}(\mathbf{u}_j'\mathbf{Y})\mathbf{u}_j .
$$

**Proof.** Let $\boldsymbol{\beta}^\star=\mathbf{V}(\mathbf{D}^2+\lambda\mathbf{I}_r)^{-1}\mathbf{D}\mathbf{U}'\mathbf{Y}$, the first sum. Since $\mathbf{X}'\mathbf{X}=\mathbf{V}\mathbf{D}^2\mathbf{V}'$ and $\mathbf{V}'\mathbf{V}=\mathbf{I}_r$,

$$
(\mathbf{X}'\mathbf{X}+\lambda\mathbf{I}_p)\boldsymbol{\beta}^\star=\mathbf{V}(\mathbf{D}^2+\lambda\mathbf{I}_r)(\mathbf{D}^2+\lambda\mathbf{I}_r)^{-1}\mathbf{D}\mathbf{U}'\mathbf{Y}
=\mathbf{V}\mathbf{D}\mathbf{U}'\mathbf{Y}=\mathbf{X}'\mathbf{Y} ,
$$

and $\mathbf{X}'\mathbf{Y}=\mathbf{X}'\mathbf{Y}_c$ because $\mathbf{1}'\mathbf{X}=\mathbf{0}'$. So $\boldsymbol{\beta}^\star$ satisfies the equation that defines $\hat{\boldsymbol{\beta}}_\lambda$ in Theorem 6.3, and $\mathbf{X}\boldsymbol{\beta}^\star=\mathbf{U}\mathbf{D}(\mathbf{D}^2+\lambda\mathbf{I}_r)^{-1}\mathbf{D}\mathbf{U}'\mathbf{Y}$ is the second sum. ∎

**Corollary 6.6** (Ridge shrinks the principal component coefficients; ISLR §6.2.1). With $\mathbf{z}_j=\mathbf{X}\mathbf{v}_j=d_j\mathbf{u}_j$ the principal component scores (Unit 5's proposition **principal components through the singular value decomposition**) and $\hat\theta_j=\mathbf{z}_j'\mathbf{Y}/\|\mathbf{z}_j\|^2$ the slopes of Unit 5's theorem **the PCR estimator**,

$$
\mathbf{X}\hat{\boldsymbol{\beta}}_\lambda=\sum_{j=1}^r\frac{d_j^2}{d_j^2+\lambda}\,\hat\theta_j\mathbf{z}_j ,\qquad
\text{while least squares fits }\sum_{j=1}^r\hat\theta_j\mathbf{z}_j .
$$

**Proof.** $\mathbf{u}_j=\mathbf{z}_j/d_j$ and $\|\mathbf{z}_j\|^2=d_j^2$, so $(\mathbf{u}_j'\mathbf{Y})\mathbf{u}_j=\hat\theta_j\mathbf{z}_j$ in Theorem 6.5. The least squares fit is the projection of $\mathbf{Y}$ onto $\mathcal{C}(\mathbf{X})=\mathcal{C}(\mathbf{z}_1,\ldots,\mathbf{z}_r)$ along the orthogonal $\mathbf{z}_j$, which is $\sum_j\hat\theta_j\mathbf{z}_j$, as in Step 1 of the proof of Unit 5's theorem **the PCR estimator**. ∎

**Remark** (ISLR §6.2.1). The factor $d_j^2/(d_j^2+\lambda)$ lies in $(0,1)$ and is smallest for the smallest $d_j$: ridge shrinks most the directions in which the predictors vary least, where least squares is least precise. PCR keeps the first $M$ directions whole and drops the rest; ridge shrinks all of them smoothly, and sets none exactly to zero.

**Proposition 6.7** (The size of the ridge estimate; ISLR §6.2.1). In the setting of Theorem 6.5, $\|\hat{\boldsymbol{\beta}}_\lambda\|^2=\sum_{j\leqslant r}\{d_j/(d_j^2+\lambda)\}^2(\mathbf{u}_j'\mathbf{Y})^2$. If $\mathbf{X}'\mathbf{Y}\neq\mathbf{0}$, it is strictly decreasing in $\lambda>0$, tends to $0$ as $\lambda\to\infty$, and tends to $\sum_j(\mathbf{u}_j'\mathbf{Y})^2/d_j^2$ as $\lambda\downarrow0$, which is $\|\hat{\boldsymbol{\beta}}\|^2$ for the least squares estimate when $\operatorname{rank}(\mathbf{X})=p$. In that case $\|\hat{\boldsymbol{\beta}}_\lambda\|/\|\hat{\boldsymbol{\beta}}\|$ decreases from $1$ to $0$.

**Proof.** The $\mathbf{v}_j$ are orthonormal, so the squared length of the first sum of Theorem 6.5 is the sum of the squared coefficients. Each $\{d_j/(d_j^2+\lambda)\}^2$ is strictly decreasing in $\lambda$ with limits $0$ and $d_j^{-2}$. $\mathbf{X}'\mathbf{Y}=\sum_jd_j(\mathbf{u}_j'\mathbf{Y})\mathbf{v}_j$ is nonzero iff some $\mathbf{u}_j'\mathbf{Y}\neq0$, and that term makes the sum strictly decreasing. When $\operatorname{rank}(\mathbf{X})=p$, $r=p$ and the least squares estimate is $\mathbf{V}\mathbf{D}^{-1}\mathbf{U}'\mathbf{Y}$ (Step 5 of the proof of Unit 5's theorem **the PCR estimator**), of squared length $\sum_j(\mathbf{u}_j'\mathbf{Y})^2/d_j^2$. ∎

**Proposition 6.8** (Conditioning). If $\operatorname{rank}(\mathbf{X})=p$ and $\lambda>0$, the condition number of $\mathbf{X}'\mathbf{X}+\lambda\mathbf{I}$ is $(d_1^2+\lambda)/(d_p^2+\lambda)$, at most that of $\mathbf{X}'\mathbf{X}$, $d_1^2/d_p^2$, and strictly smaller when $d_1>d_p$.

**Proof.** $\mathbf{X}'\mathbf{X}+\lambda\mathbf{I}=\mathbf{V}(\mathbf{D}^2+\lambda\mathbf{I})\mathbf{V}'$ with $\mathbf{V}$ orthogonal is symmetric positive definite with eigenvalues $d_j^2+\lambda$, so its condition number is the ratio of the extreme ones by the prerequisites' proposition on the operator norm. For $a>b>0$, $t\mapsto(a+t)/(b+t)$ has derivative $(b-a)/(b+t)^2<0$, so by the prerequisites' mean value theorem it decreases strictly. ∎

### 1.3 Why shrinkage can help

**Proposition 6.9** (Bias and variance of ridge; ISLR §6.2.1). Let $\mathbf{Y}=\beta_0\mathbf{1}+\mathbf{X}\boldsymbol{\beta}+\boldsymbol{\varepsilon}$ with $\mathbf{X}$ centred, $\operatorname{\mathsf{E}}[\boldsymbol{\varepsilon}]=\mathbf{0}$ and $\operatorname{var}(\boldsymbol{\varepsilon})=\sigma^2\mathbf{I}_n$, and let $\mathbf{A}_\lambda=\mathbf{X}'\mathbf{X}+\lambda\mathbf{I}$, $\lambda>0$. Then

$$
\operatorname{\mathsf{E}}[\hat{\boldsymbol{\beta}}_\lambda]-\boldsymbol{\beta}=-\lambda\mathbf{A}_\lambda^{-1}\boldsymbol{\beta} ,\qquad
\operatorname{var}(\hat{\boldsymbol{\beta}}_\lambda)=\sigma^2\mathbf{A}_\lambda^{-1}\mathbf{X}'\mathbf{X}\mathbf{A}_\lambda^{-1} .
$$

The same holds for $\mathbf{Y}=\mathbf{X}\boldsymbol{\beta}+\boldsymbol{\varepsilon}$ without intercept, any $\mathbf{X}$, and $\hat{\boldsymbol{\beta}}_\lambda=\mathbf{A}_\lambda^{-1}\mathbf{X}'\mathbf{Y}$.

**Proof.** $\hat{\boldsymbol{\beta}}_\lambda=\mathbf{A}_\lambda^{-1}\mathbf{X}'\mathbf{Y}$, as $\mathbf{X}'\mathbf{Y}_c=\mathbf{X}'\mathbf{Y}$, is linear in $\mathbf{Y}$, and $\operatorname{\mathsf{E}}[\mathbf{X}'\mathbf{Y}]=\mathbf{X}'\mathbf{X}\boldsymbol{\beta}$ because $\mathbf{X}'\mathbf{1}=\mathbf{0}$, or directly without intercept; so $\operatorname{\mathsf{E}}[\hat{\boldsymbol{\beta}}_\lambda]=\mathbf{A}_\lambda^{-1}(\mathbf{A}_\lambda-\lambda\mathbf{I})\boldsymbol{\beta}$. The variance follows from the prerequisites' proposition **variance under a linear map**. ∎

**Theorem 6.10** (Some ridge beats least squares; ISLR §6.2.1). In the setting of Proposition 6.9, let $\operatorname{rank}(\mathbf{X})=p$, $\sigma^2>0$ and $\boldsymbol{\beta}\neq\mathbf{0}$; let $\gamma_1\geqslant\cdots\geqslant\gamma_p>0$ be the eigenvalues of $\mathbf{X}'\mathbf{X}$ with orthonormal eigenvectors $\mathbf{v}_j$, $\alpha_j=\mathbf{v}_j'\boldsymbol{\beta}$, and $R(\lambda)=\operatorname{\mathsf{E}}\|\hat{\boldsymbol{\beta}}_\lambda-\boldsymbol{\beta}\|^2$, with $\hat{\boldsymbol{\beta}}_0$ the least squares estimate. Then

$$
R(\lambda)=\sum_{j=1}^p\frac{\sigma^2\gamma_j}{(\gamma_j+\lambda)^2}
+\sum_{j=1}^p\frac{\lambda^2\alpha_j^2}{(\gamma_j+\lambda)^2},\qquad
R'(\lambda)=\sum_{j=1}^p\frac{2\gamma_j(\lambda\alpha_j^2-\sigma^2)}{(\gamma_j+\lambda)^3} .
$$

The variance sum decreases and the squared bias sum increases in $\lambda$; $R$ decreases strictly on $[0,\sigma^2/\max_j\alpha_j^2)$, increases strictly for all large $\lambda$, and $R(\lambda)\to\|\boldsymbol{\beta}\|^2$ as $\lambda\to\infty$. In particular $R(\lambda)<R(0)$ for some $\lambda>0$.

**Proof.** **Step 1: the formula.** By the spectral theorem of the prerequisites, $\mathbf{X}'\mathbf{X}=\mathbf{V}\boldsymbol{\Gamma}\mathbf{V}'$ with $\boldsymbol{\Gamma}=\operatorname{diag}(\gamma_j)$ and $\mathbf{V}$ orthogonal, so $\mathbf{A}_\lambda^{-1}=\mathbf{V}(\boldsymbol{\Gamma}+\lambda\mathbf{I})^{-1}\mathbf{V}'$. Expanding each square about $\operatorname{\mathsf{E}}[\hat\beta_{\lambda j}]$, the cross term having mean zero, $\operatorname{\mathsf{E}}(\hat\beta_{\lambda j}-\beta_j)^2=\operatorname{var}(\hat\beta_{\lambda j})+(\operatorname{\mathsf{E}}[\hat\beta_{\lambda j}]-\beta_j)^2$, and summing over $j$, $R(\lambda)=\operatorname{tr}\operatorname{var}(\hat{\boldsymbol{\beta}}_\lambda)+\|\operatorname{\mathsf{E}}[\hat{\boldsymbol{\beta}}_\lambda]-\boldsymbol{\beta}\|^2$. By Proposition 6.9 and the prerequisites' proposition **properties of the trace**, $\operatorname{tr}\operatorname{var}(\hat{\boldsymbol{\beta}}_\lambda)=\sigma^2\operatorname{tr}\{(\boldsymbol{\Gamma}+\lambda\mathbf{I})^{-2}\boldsymbol{\Gamma}\}=\sum_j\sigma^2\gamma_j/(\gamma_j+\lambda)^2$, and $\|\lambda\mathbf{A}_\lambda^{-1}\boldsymbol{\beta}\|^2=\lambda^2\|(\boldsymbol{\Gamma}+\lambda\mathbf{I})^{-1}\mathbf{V}'\boldsymbol{\beta}\|^2 =\sum_j\lambda^2\alpha_j^2/(\gamma_j+\lambda)^2$. At $\lambda=0$ the formula is $\sigma^2\sum_j\gamma_j^{-1}=\sigma^2\operatorname{tr}(\mathbf{X}'\mathbf{X})^{-1}$, the risk of the unbiased least squares estimate by Unit 2's theorems **unbiasedness** and **variance–covariance matrix**: $\tilde{\mathbf{X}}=[\mathbf{1}\ \mathbf{X}]$ has $\tilde{\mathbf{X}}'\tilde{\mathbf{X}}=\operatorname{diag}(n,\mathbf{X}'\mathbf{X})$ for centred $\mathbf{X}$, so the slope block of $\sigma^2(\tilde{\mathbf{X}}'\tilde{\mathbf{X}})^{-1}$ is $\sigma^2(\mathbf{X}'\mathbf{X})^{-1}$; without intercept the theorems apply to $\mathbf{X}$ itself.

**Step 2: the derivative.** The two summands have derivatives $-2\sigma^2\gamma_j/(\gamma_j+\lambda)^3<0$ and $\{2\lambda\alpha_j^2(\gamma_j+\lambda)-2\lambda^2\alpha_j^2\}/(\gamma_j+\lambda)^3 =2\lambda\alpha_j^2\gamma_j/(\gamma_j+\lambda)^3$, positive for $\lambda>0$ when $\alpha_j\neq0$; their sum is $R'$. As $\boldsymbol{\beta}\neq\mathbf{0}$, some $\alpha_j\neq0$, so by the prerequisites' mean value theorem the variance sum decreases strictly and the squared bias sum increases strictly.

**Step 3: the two ends.** Every summand of $R'$ is negative when $\lambda\alpha_j^2<\sigma^2$ for all $j$. For large $\lambda$,

$$
\lambda^3R'(\lambda)\geqslant2\lambda\Bigl(\frac{\lambda}{\gamma_1+\lambda}\Bigr)^3\sum_j\gamma_j\alpha_j^2-2\sigma^2\sum_j\gamma_j ,
$$

using $\lambda/(\gamma_j+\lambda)\geqslant\lambda/(\gamma_1+\lambda)$ and $\lambda/(\gamma_j+\lambda)\leqslant1$; the right side tends to $\infty$ because $\sum_j\gamma_j\alpha_j^2=\boldsymbol{\beta}'\mathbf{X}'\mathbf{X}\boldsymbol{\beta}>0$. The limit of $R$ is $\sum_j\alpha_j^2=\|\boldsymbol{\beta}\|^2$, the first sum tending to $0$. By the prerequisites' mean value theorem a negative derivative on an interval makes $R$ strictly decreasing there, and a positive one strictly increasing. ∎

**Remark** (ISLR §6.2.1). For a new observation $y_0=\beta_0+\mathbf{x}_0'\boldsymbol{\beta}+\varepsilon_0$, with $\mathbf{x}_0$ and $\varepsilon_0$ independent of each other and of the training data, $\operatorname{\mathsf{E}}[\mathbf{x}_0]=\mathbf{0}$, $\operatorname{\mathsf{E}}[\mathbf{x}_0\mathbf{x}_0']=\mathbf{I}$, $\operatorname{\mathsf{E}}[\varepsilon_0]=0$ and $\operatorname{var}(\varepsilon_0)=\sigma^2$, the expected test error of $\hat y_0=\hat\beta_0+\mathbf{x}_0'\hat{\boldsymbol{\beta}}_\lambda$ is $\sigma^2+\operatorname{\mathsf{E}}(\hat\beta_0-\beta_0)^2+R(\lambda)$: in $y_0-\hat y_0=\varepsilon_0+(\beta_0-\hat\beta_0)+\mathbf{x}_0'(\boldsymbol{\beta}-\hat{\boldsymbol{\beta}}_\lambda)$ the cross terms have mean zero by the independence and the zero means, and $\operatorname{\mathsf{E}}[\{\mathbf{x}_0'(\boldsymbol{\beta}-\hat{\boldsymbol{\beta}}_\lambda)\}^2]=\operatorname{\mathsf{E}}\|\boldsymbol{\beta}-\hat{\boldsymbol{\beta}}_\lambda\|^2$ because $\operatorname{\mathsf{E}}[\mathbf{x}_0\mathbf{x}_0']=\mathbf{I}$. So $R$ is the part of the test error that $\lambda$ controls. At $\lambda=0$, $R=\sigma^2\sum_j\gamma_j^{-1}$ and $R'=-2\sigma^2\sum_j\gamma_j^{-2}$ are large in size when some $\gamma_j=\|\mathbf{X}\mathbf{v}_j\|^2$ is small, that is, when a combination of the predictors is nearly zero. The interval on which $R$ decreases depends on the unknown $\sigma^2$ and $\boldsymbol{\beta}$, so $\lambda$ is chosen from the data.

**Remark** (ISLR §6.2.1). Ridge regression also has a computational advantage over best subset selection, which searches $2^p$ models. For a given $\lambda$ ridge fits a single model, and the fits for all $\lambda$ together cost about as much as one least squares fit: by the singular value decomposition of §1.2 every $\hat{\boldsymbol{\beta}}_\lambda$ comes from the same decomposition of $\mathbf{X}$.

### 1.4 Effective degrees of freedom (beyond the syllabus)

> **Reading.** Beyond the syllabus: ISLR §6.4.3 measures the flexibility of the lasso by its number of nonzero coefficients; this subsection gives the corresponding measure for ridge.

**Definition 6.11** (Linear smoother and effective degrees of freedom). A fitting rule is a **linear smoother** if its fitted values are $\hat{\mathbf{Y}}=\mathbf{S}\mathbf{Y}$ for a matrix $\mathbf{S}$ that does not depend on $\mathbf{Y}$; its **effective degrees of freedom** are $\operatorname{tr}(\mathbf{S})$.

**Proposition 6.12** (Degrees of freedom of ridge). With centred $\mathbf{X}$, the ridge fit is the linear smoother with $\mathbf{S}_\lambda=\mathbf{1}\mathbf{1}'/n+\mathbf{X}(\mathbf{X}'\mathbf{X}+\lambda\mathbf{I})^{-1}\mathbf{X}'$, and

$$
\operatorname{df}(\lambda)=\operatorname{tr}(\mathbf{S}_\lambda)=1+\sum_{j=1}^r\frac{d_j^2}{d_j^2+\lambda} ,
$$

which, if $r\geqslant1$, decreases strictly from $1+r$ as $\lambda\downarrow0$ to $1$ as $\lambda\to\infty$.

**Proof.** $\hat\beta_0\mathbf{1}+\mathbf{X}\hat{\boldsymbol{\beta}}_\lambda=\bar y\mathbf{1}+\mathbf{X}(\mathbf{X}'\mathbf{X}+\lambda\mathbf{I})^{-1}\mathbf{X}'\mathbf{Y}$ by Proposition 6.2 and Theorem 6.3. By Theorem 6.5, $\mathbf{X}(\mathbf{X}'\mathbf{X}+\lambda\mathbf{I})^{-1}\mathbf{X}'=\sum_j\{d_j^2/(d_j^2+\lambda)\}\mathbf{u}_j\mathbf{u}_j'$, and $\operatorname{tr}(\mathbf{u}_j\mathbf{u}_j')=\mathbf{u}_j'\mathbf{u}_j=1$, $\operatorname{tr}(\mathbf{1}\mathbf{1}'/n)=1$. Each summand decreases strictly from $1$ to $0$. ∎

**Theorem 6.13** (The optimism of a linear smoother). Let $\mathbf{Y}=\boldsymbol{\mu}+\boldsymbol{\varepsilon}$ with $\operatorname{\mathsf{E}}[\boldsymbol{\varepsilon}]=\mathbf{0}$ and $\operatorname{var}(\boldsymbol{\varepsilon})=\sigma^2\mathbf{I}_n$, $\mathbf{Y}^0=\boldsymbol{\mu}+\boldsymbol{\varepsilon}^0$ with $\boldsymbol{\varepsilon}^0$ independent of $\boldsymbol{\varepsilon}$ and distributed as $\boldsymbol{\varepsilon}$, and $\hat{\mathbf{Y}}=\mathbf{S}\mathbf{Y}$. Then

$$
\operatorname{\mathsf{E}}\|\mathbf{Y}^0-\hat{\mathbf{Y}}\|^2-\operatorname{\mathsf{E}}\|\mathbf{Y}-\hat{\mathbf{Y}}\|^2=2\sigma^2\operatorname{tr}(\mathbf{S}) .
$$

**Proof.** With $\mathbf{M}=\mathbf{I}-\mathbf{S}$, the prerequisites' lemma **expectation of a quadratic form** gives $\operatorname{\mathsf{E}}\|\mathbf{M}\mathbf{Y}\|^2=\sigma^2\operatorname{tr}(\mathbf{M}'\mathbf{M})+\|\mathbf{M}\boldsymbol{\mu}\|^2$. As in Step 3 of the proof of Unit 5's theorem **the optimism of the training error**, independence and $\operatorname{\mathsf{E}}[\boldsymbol{\varepsilon}^0]=\mathbf{0}$ give $\operatorname{\mathsf{E}}\|\mathbf{Y}^0-\mathbf{S}\mathbf{Y}\|^2=n\sigma^2+\operatorname{\mathsf{E}}\|\mathbf{M}\boldsymbol{\mu}-\mathbf{S}\boldsymbol{\varepsilon}\|^2 =n\sigma^2+\|\mathbf{M}\boldsymbol{\mu}\|^2+\sigma^2\operatorname{tr}(\mathbf{S}'\mathbf{S})$, the cross term having mean zero. Since $\operatorname{tr}(\mathbf{M}'\mathbf{M})=n-2\operatorname{tr}(\mathbf{S})+\operatorname{tr}(\mathbf{S}'\mathbf{S})$, the difference is $2\sigma^2\operatorname{tr}(\mathbf{S})$. ∎

**Remark**. With $\mathbf{S}=\mathbf{H}_1$, a projection of rank $q$, this is Unit 5's theorem **the optimism of the training error**; for ridge the charge is $2\sigma^2\operatorname{df}(\lambda)$, less than the $2\sigma^2(1+r)$ of least squares, $1+r=\operatorname{rank}[\mathbf{1}\ \mathbf{X}]$, when $r=\operatorname{rank}(\mathbf{X})\geqslant1$.

### 1.5 Ridge on the Credit data

**Example 6.1** (Ridge coefficient paths; ISLR §6.2.1, Figure 6.4). The standardised ridge coefficients of `Balance` on the eleven `Credit` predictors, through Theorem 6.5:

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 2.5)
Xc <- model.matrix(Balance ~ ., Credit)[, -1]; yc <- Credit$Balance; n_c <- nrow(Xc); p_c <- ncol(Xc)
sdn <- function(v) sqrt(mean((v - mean(v))^2))
Xs_c <- scale(Xc, center = TRUE, scale = apply(Xc, 2, sdn))
sv_c <- svd(Xs_c); uy_c <- drop(crossprod(sv_c$u, yc))
ridge_c <- function(l) drop(sv_c$v %*% (sv_c$d / (sv_c$d^2 + l) * uy_c))
lam_c <- 10^seq(-2, 5, length = 300)
B_c <- sapply(lam_c, ridge_c); rownames(B_c) <- colnames(Xc)
b_ls <- setNames(ridge_c(0), colnames(Xc)); ratio_c <- sqrt(colSums(B_c^2)) / sqrt(sum(b_ls^2))
round(b_ls[c("Income", "Limit", "Rating", "StudentYes")], 2)
c(ratio_strictly_decreasing = all(diff(ratio_c) < 0), abs_Limit_strictly_decreasing = all(diff(abs(B_c["Limit", ])) < 0))
round(c(lambda_max_abs_Rating = lam_c[which.max(abs(B_c["Rating", ]))], max_abs_Rating = max(abs(B_c["Rating", ])),
  cor_Limit_Rating = cor(Xc[, "Limit"], Xc[, "Rating"])), 3)
col_c <- ifelse(colnames(Xc) %in% c("Income", "Limit", "Rating", "StudentYes"),
                c(Income = 1, Limit = 2, Rating = 4, StudentYes = 3)[colnames(Xc)], 8)
op <- par(mfrow = c(1, 2), mar = c(3.2, 3.8, 0.6, 0.6), mgp = c(2.1, 0.6, 0))
ax10 <- function() axis(1, at = 10^c(-2, 0, 2, 4), labels = c("0.01", "1", "100", "10000"))
matplot(lam_c, t(B_c), type = "l", lty = 1, log = "x", col = col_c, xlab = "lambda",
        ylab = "standardised coefficient", xaxt = "n"); ax10()
legend("topright", c("Income", "Limit", "Rating", "StudentYes"), col = c(1, 2, 4, 3), lty = 1, bty = "n", cex = 0.7)
matplot(ratio_c, t(B_c), type = "l", lty = 1, col = col_c, xlab = "norm ratio", ylab = "")
par(op)

*Figure 6.1.* Standardised ridge coefficients for `Credit` against $\lambda$ (log scale) and against $\|\hat{\boldsymbol{\beta}}_\lambda\|/\|\hat{\boldsymbol{\beta}}\|$ (ISLR Figure 6.4).

The least squares standardised coefficients of `Income`, `Limit`, `Rating` and `StudentYes` are $-274.7$, $440.1$, $175.6$ and $127.7$. The norm ratio decreases strictly, as Proposition 6.7 requires, but a single coefficient need not shrink: $|\hat\beta_{\texttt{Rating}}|$ rises from $175.6$ to $281.805$ at $\lambda=7.579$ before tending to zero, while $|\hat\beta_{\texttt{Limit}}|$, whose predictor has correlation $0.997$ with `Rating`, decreases strictly from $440.1$.

**Example 6.2** (Bias, variance and test error of ridge; ISLR §6.2.1, Figure 6.5). $n=50$ observations of $p=45$ independent standard normal predictors, fixed, with $\boldsymbol{\beta}=(0.5,\ldots,0.5)'$, $\sigma=1$ and no intercept; for test points $\mathbf{x}_0$ with independent standard normal coordinates the expected test error is $\sigma^2+\|\operatorname{\mathsf{E}}\hat{\boldsymbol{\beta}}_\lambda-\boldsymbol{\beta}\|^2+\operatorname{tr}\operatorname{var}(\hat{\boldsymbol{\beta}}_\lambda)$, by the remark after Theorem 6.10 without the intercept, computed exactly from Proposition 6.9:

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 2.4)
set.seed(645); n_s <- 50; p_s <- 45; X_s <- matrix(rnorm(n_s * p_s), n_s); XtX_s <- crossprod(X_s)
b_dense <- rep(0.5, p_s); b_sparse <- c(2.5, 2.5, rep(0, p_s - 2)); lam_s <- 10^seq(3, -2, length = 60)
ridge_exact <- function(beta) t(sapply(lam_s, function(l) { W <- solve(XtX_s + l * diag(p_s), t(X_s))
  c(bias2 = sum((W %*% X_s %*% beta - beta)^2), var = sum(W^2)) }))
re_d <- ridge_exact(b_dense); mse_rd <- 1 + rowSums(re_d)
c(min_test_MSE = min(mse_rd), at_lambda = lam_s[which.min(mse_rd)],
  least_squares = 1 + sum(diag(solve(XtX_s))), null_model = 1 + sum(b_dense^2))
op <- par(mar = c(3.2, 3.8, 0.6, 0.6), mgp = c(2.1, 0.6, 0))
matplot(lam_s, cbind(re_d, mse_rd), type = "l", lty = 1, lwd = 1.5, log = "x", col = c(1, 3, 6),
        xlab = "lambda", ylab = "", ylim = c(0, 16))
legend("topright", c("squared bias", "variance", "test MSE"), col = c(1, 3, 6), lty = 1, bty = "n", cex = 0.7)
par(op)

*Figure 6.2.* Squared bias, variance and expected test error of ridge regression against $\lambda$ for the simulated design (ISLR Figure 6.5).

The expected test error is smallest, $3.376$, at $\lambda=3.486$. Least squares, which is unbiased, has $17.158$, more than the null model's $12.250$: its variance $\sigma^2\operatorname{tr}(\mathbf{X}'\mathbf{X})^{-1}$ alone exceeds the error of the null model. The variance falls and the squared bias rises with $\lambda$, as Theorem 6.10 states.

## 2 The Lasso

> **Reading.** ISLR §6.2.2. SRM learning outcome 2(g).

### 2.1 Definition, existence and the constrained form

**Definition 6.14** (The lasso; ISLR §6.2.2). With $\mathbf{X}$ and $\mathbf{Y}$ as in Definition 6.1, the **lasso** estimates minimise $\mathrm{RSS}+\lambda\|\boldsymbol{\beta}\|_1$, $\|\boldsymbol{\beta}\|_1=\sum_j|\beta_j|$, over $(\beta_0,\boldsymbol{\beta})$. With centred predictors, the argument of Proposition 6.2, which does not use the form of the penalty, gives $\hat\beta_0=\bar y$ and leaves the criterion

$$
L_\lambda(\boldsymbol{\beta})=\|\mathbf{Y}_c-\mathbf{X}\boldsymbol{\beta}\|^2+\lambda\|\boldsymbol{\beta}\|_1 ,
$$

whose minimisers are the **lasso estimators** $\hat{\boldsymbol{\beta}}$. The **active set** of $\boldsymbol{\beta}$ is $A(\boldsymbol{\beta})=\{j:\beta_j\neq0\}$, and a model is **sparse** when its active set is small.

**Theorem 6.15** (Existence, and one fit; ISLR §6.2.2). For $\lambda>0$, $L_\lambda$ attains its minimum. Any two minimisers $\hat{\boldsymbol{\beta}}$, $\hat{\boldsymbol{\beta}}'$ have $\mathbf{X}\hat{\boldsymbol{\beta}}=\mathbf{X}\hat{\boldsymbol{\beta}}'$ and $\|\hat{\boldsymbol{\beta}}\|_1=\|\hat{\boldsymbol{\beta}}'\|_1$.

**Proof.** $L_\lambda$ is continuous and $L_\lambda(\boldsymbol{\beta})\geqslant\lambda\|\boldsymbol{\beta}\|_1\geqslant\lambda\|\boldsymbol{\beta}\|$, because $\|\boldsymbol{\beta}\|_1^2=\|\boldsymbol{\beta}\|^2+2\sum_{i<j}|\beta_i||\beta_j|$; so $L_\lambda$ is coercive and attains its minimum by the prerequisites' lemma **coercive minimisation**. Let $\hat{\boldsymbol{\beta}},\hat{\boldsymbol{\beta}}'$ be minimisers with value $m$, suppose $\mathbf{X}\hat{\boldsymbol{\beta}}\neq\mathbf{X}\hat{\boldsymbol{\beta}}'$, and put $\boldsymbol{\beta}_t=t\hat{\boldsymbol{\beta}}+(1-t)\hat{\boldsymbol{\beta}}'$, $0<t<1$. With $\mathbf{w}=\mathbf{X}\hat{\boldsymbol{\beta}}-\mathbf{X}\hat{\boldsymbol{\beta}}'\neq\mathbf{0}$,

$$
t\|\mathbf{Y}_c-\mathbf{X}\hat{\boldsymbol{\beta}}\|^2+(1-t)\|\mathbf{Y}_c-\mathbf{X}\hat{\boldsymbol{\beta}}'\|^2-\|\mathbf{Y}_c-\mathbf{X}\boldsymbol{\beta}_t\|^2=t(1-t)\|\mathbf{w}\|^2>0 ,
$$

by expanding the squares, while $\|\boldsymbol{\beta}_t\|_1\leqslant t\|\hat{\boldsymbol{\beta}}\|_1+(1-t)\|\hat{\boldsymbol{\beta}}'\|_1$ by the triangle inequality. Adding, $L_\lambda(\boldsymbol{\beta}_t)<m$, which is impossible. So $\mathbf{X}\hat{\boldsymbol{\beta}}=\mathbf{X}\hat{\boldsymbol{\beta}}'$, the two residual sums of squares agree, and so do the penalties. ∎

**Theorem 6.16** (The penalised and the constrained forms; ISLR §6.2.2). Let $P:\mathbb{R}^p\to[0,\infty)$ be convex with $P(\mathbf{0})=0$, and $G(\boldsymbol{\beta})=\|\mathbf{Y}_c-\mathbf{X}\boldsymbol{\beta}\|^2$.

(i) If $\hat{\boldsymbol{\beta}}$ minimises $G+\lambda P$ for some $\lambda\geqslant0$, it minimises $G$ subject to $P(\boldsymbol{\beta})\leqslant P(\hat{\boldsymbol{\beta}})$.

(ii) If $s>0$ and $\hat{\boldsymbol{\beta}}$ minimises $G$ subject to $P(\boldsymbol{\beta})\leqslant s$, then $\hat{\boldsymbol{\beta}}$ minimises $G+\lambda P$ for some $\lambda\geqslant0$.

With $P=\|\cdot\|_1$ these are the two forms of the lasso, and with $P=\|\cdot\|^2$ those of ridge: minimising $\mathrm{RSS}$ over the diamond $\|\boldsymbol{\beta}\|_1\leqslant s$, or over the ball $\|\boldsymbol{\beta}\|^2\leqslant s$.

**Proof.** (i) If $P(\boldsymbol{\beta})\leqslant P(\hat{\boldsymbol{\beta}})$, then $G(\hat{\boldsymbol{\beta}})+\lambda P(\hat{\boldsymbol{\beta}})\leqslant G(\boldsymbol{\beta})+\lambda P(\boldsymbol{\beta})\leqslant G(\boldsymbol{\beta})+\lambda P(\hat{\boldsymbol{\beta}})$, and cancelling gives $G(\hat{\boldsymbol{\beta}})\leqslant G(\boldsymbol{\beta})$.

(ii) The problem of minimising $f_0=G$ subject to $f_1(\boldsymbol{\beta})=P(\boldsymbol{\beta})-s\leqslant0$ over $\mathbb{R}^p$ is a convex programme of the prerequisites, with finite value $G(\hat{\boldsymbol{\beta}})$, and Slater's condition holds at $\mathbf{0}$, where $f_1(\mathbf{0})=-s<0$. By the prerequisites' theorem **Slater implies strong duality**, the dual optimum is attained at some $\lambda\geqslant0$:

$$
G(\hat{\boldsymbol{\beta}})=\inf_{\boldsymbol{\beta}}\{G(\boldsymbol{\beta})+\lambda(P(\boldsymbol{\beta})-s)\}\leqslant G(\hat{\boldsymbol{\beta}})+\lambda(P(\hat{\boldsymbol{\beta}})-s)
\leqslant G(\hat{\boldsymbol{\beta}}) ,
$$

the last step because $\lambda\geqslant0$ and $P(\hat{\boldsymbol{\beta}})\leqslant s$. Both inequalities are equalities: $\lambda(P(\hat{\boldsymbol{\beta}})-s)=0$, and then $\inf_{\boldsymbol{\beta}}\{G(\boldsymbol{\beta})+\lambda P(\boldsymbol{\beta})\}=G(\hat{\boldsymbol{\beta}})+\lambda P(\hat{\boldsymbol{\beta}})$. ∎

**Proposition 6.17** (Best subset selection as a constrained problem; ISLR §6.2.2). For an integer $0\leqslant s\leqslant p$, minimising $\|\mathbf{Y}_c-\mathbf{X}\boldsymbol{\beta}\|^2$ subject to $\sum_j\mathbb{1}\{\beta_j\neq0\}\leqslant s$ gives the residual sum of squares of Unit 5's best subset model $\mathcal{M}_s$ (the predictors centred).

**Proof.** The feasible $\boldsymbol{\beta}$ are those supported on some $S$ with $|S|\leqslant s$, so the minimum is $\min_{|S|\leqslant s}\mathrm{RSS}(S)$, which is $\mathrm{RSS}(\mathcal{M}_s)$ because the residual sum of squares of the best subset model does not increase with its size (Unit 5's proposition **counting and nesting**); with centred predictors the intercept is $\bar y$ in every fit, by Proposition 6.2 at $\lambda=0$. ∎

### 2.2 Optimality conditions and sparsity

**Theorem 6.18** (Stationarity for the lasso). Let $\lambda>0$ and $\mathbf{c}=\mathbf{X}'(\mathbf{Y}_c-\mathbf{X}\hat{\boldsymbol{\beta}})$. Then $\hat{\boldsymbol{\beta}}$ minimises $L_\lambda$ iff

$$
2c_j=\lambda\operatorname{sign}(\hat\beta_j)\ \text{ for }\hat\beta_j\neq0,\qquad
|2c_j|\leqslant\lambda\ \text{ for }\hat\beta_j=0 .
$$

**Proof.** $L_\lambda$ is convex, so by the prerequisites' lemma **directional characterisation of a convex minimum**, $\hat{\boldsymbol{\beta}}$ is a minimiser iff $L'_\lambda(\hat{\boldsymbol{\beta}};\mathbf{v})\geqslant0$ for every $\mathbf{v}$. The sum of squares is differentiable with gradient $-2\mathbf{X}'(\mathbf{Y}_c-\mathbf{X}\boldsymbol{\beta})$ and contributes $-2\mathbf{c}'\mathbf{v}$. For the penalty, $\{|\hat\beta_j+tv_j|-|\hat\beta_j|\}/t$ equals $v_j\operatorname{sign}(\hat\beta_j)$ for small $t>0$ when $\hat\beta_j\neq0$, and $|v_j|$ when $\hat\beta_j=0$. So

$$
L'_\lambda(\hat{\boldsymbol{\beta}};\mathbf{v})=\sum_{\hat\beta_j\neq0}\{\lambda\operatorname{sign}(\hat\beta_j)-2c_j\}v_j
+\sum_{\hat\beta_j=0}\{\lambda|v_j|-2c_jv_j\} .
$$

At a minimiser, $\mathbf{v}=\pm\mathbf e_j$ gives $\pm\{\lambda\operatorname{sign}(\hat\beta_j)-2c_j\}\geqslant0$ when $\hat\beta_j\neq0$, forcing equality, and $\lambda\mp2c_j\geqslant0$ when $\hat\beta_j=0$. Conversely, under the conditions the first sum vanishes and each term of the second is at least $\lambda|v_j|-|2c_j||v_j|\geqslant0$. ∎

**Corollary 6.19** (The smallest $\lambda$ with the null model; ISLR §6.2.2). $\mathbf{0}$ is a lasso estimator iff $\lambda\geqslant\lambda_{\max}=2\max_j|\mathbf{x}_j'\mathbf{Y}_c|$.

**Proof.** At $\hat{\boldsymbol{\beta}}=\mathbf{0}$, $\mathbf{c}=\mathbf{X}'\mathbf{Y}_c$ and every coordinate is zero, so the conditions of Theorem 6.18 read $|2\mathbf{x}_j'\mathbf{Y}_c|\leqslant\lambda$ for all $j$. ∎

**Theorem 6.20** (Sparsity; ISLR §6.2.2). For $\lambda>0$ there is a lasso estimator whose active columns are linearly independent; it has at most $\operatorname{rank}(\mathbf{X})\leqslant\min(n-1,p)$ nonzero coefficients.

**Proof.** Let $\hat{\boldsymbol{\beta}}$ be a minimiser whose active columns are dependent: $\mathbf{X}\mathbf{v}=\mathbf{0}$ for some $\mathbf{v}\neq\mathbf{0}$ supported in $A=A(\hat{\boldsymbol{\beta}})$. For $|t|$ small every $\hat\beta_j+tv_j$, $j\in A$, keeps the sign of $\hat\beta_j$, so $L_\lambda(\hat{\boldsymbol{\beta}}+t\mathbf{v})=L_\lambda(\hat{\boldsymbol{\beta}})+\lambda t\delta$ with $\delta=\sum_{j\in A}\operatorname{sign}(\hat\beta_j)v_j$; minimality at $t=0$ forces $\delta=0$. Replacing $\mathbf{v}$ by $-\mathbf{v}$ if needed, some ratio $-\hat\beta_j/v_j$, $j\in A$, $v_j\neq0$, is positive; let $t^+$ be the smallest positive one. On $[0,t^+]$ no active coordinate changes sign, so $L_\lambda$ is constant there, and $\hat{\boldsymbol{\beta}}+t^+\mathbf{v}$ is a minimiser with a strictly smaller active set. Repeating ends at a minimiser with independent active columns, of which there are at most $\operatorname{rank}(\mathbf{X})$; and $\operatorname{rank}(\mathbf{X})\leqslant n-1$ because the centred columns lie in the orthogonal complement of $\mathbf{1}$. ∎

**Remark** (ISLR §6.2.2). By Theorem 6.5 the $j$th ridge coefficient times $\prod_m(d_m^2+\lambda)$ is a polynomial in $\lambda$, so unless it vanishes for every $\lambda>0$ it vanishes for at most finitely many: ridge does not select variables. The lasso does: all its minimisers share $\mathbf{X}\hat{\boldsymbol{\beta}}$ and $\|\hat{\boldsymbol{\beta}}\|_1$ (Theorem 6.15), hence $\mathbf{c}$, so every lasso estimator is $\mathbf{0}$ for $\lambda\geqslant\lambda_{\max}$ (Corollary 6.19) and has $\hat\beta_j=0$ whenever $|2c_j|<\lambda$ (Theorem 6.18), which yields sparse models. In the constrained form the corners of the diamond lie on the axes, and the first contour of $\mathrm{RSS}$ to touch it often does so at a corner.

### 2.3 The orthonormal design

**Lemma 6.21** (Soft thresholding; ISLR §6.2.2). For $a\in\mathbb{R}$ and $\tau>0$, $h(b)=(b-a)^2+2\tau|b|$ has the unique minimiser $S_\tau(a)=\operatorname{sign}(a)(|a|-\tau)_+$: $a-\tau$ if $a>\tau$, $0$ if $|a|\leqslant\tau$, $a+\tau$ if $a<-\tau$.

**Proof.** $h$ is continuous, coercive and strictly convex, so it has one minimiser by the prerequisites' lemma **coercive minimisation**. As $h$ for $-a$ at $-b$ equals $h$ for $a$ at $b$, take $a\geqslant0$. For $b<0$, $(b-a)^2\geqslant(|b|-a)^2$, so $h(b)\geqslant h(|b|)$ and the minimiser lies in $[0,\infty)$, where $h(b)=(b-(a-\tau))^2+\text{const}$ is minimised at $\max(a-\tau,0)$. ∎

**Theorem 6.22** (Ridge and lasso with orthonormal predictors; ISLR §6.2.2). If $\mathbf{X}'\mathbf{X}=\mathbf{I}_p$ and $\hat\beta_j=\mathbf{x}_j'\mathbf{Y}_c$ are the least squares coefficients, then

$$
\hat\beta_{\lambda j}^{\,\mathrm{ridge}}=\frac{\hat\beta_j}{1+\lambda},\qquad
\hat\beta_{\lambda j}^{\,\mathrm{lasso}}=S_{\lambda/2}(\hat\beta_j) .
$$

ISLR's special case, $n=p$, $\mathbf{X}=\mathbf{I}$ and no intercept, is the same computation with $\mathbf{Y}$ in place of $\mathbf{Y}_c$ and $\hat\beta_j=y_j$.

**Proof.** With $\mathbf{X}'\mathbf{X}=\mathbf{I}$, $\hat{\boldsymbol{\beta}}=\mathbf{X}'\mathbf{Y}_c$, and $\mathbf{Y}_c-\mathbf{X}\boldsymbol{\beta}=(\mathbf{Y}_c-\mathbf{X}\hat{\boldsymbol{\beta}})+\mathbf{X}(\hat{\boldsymbol{\beta}}-\boldsymbol{\beta})$ with orthogonal terms, so by Pythagoras $\|\mathbf{Y}_c-\mathbf{X}\boldsymbol{\beta}\|^2=\|\mathbf{Y}_c-\mathbf{X}\hat{\boldsymbol{\beta}}\|^2+\sum_j(\beta_j-\hat\beta_j)^2$. Both criteria separate into $p$ scalar problems. The ridge one, $(b-\hat\beta_j)^2+\lambda b^2$, is strictly convex with derivative $2(b-\hat\beta_j)+2\lambda b$, zero at $\hat\beta_j/(1+\lambda)$; the lasso one is Lemma 6.21 with $2\tau=\lambda$. ∎

**Proposition 6.23** (Bias and variance of soft thresholding; ISLR §6.2.2). Let $z\sim N(\theta,\sigma^2)$ with $\sigma>0$, and $\tau\geqslant0$. Then $\operatorname{\mathsf{E}}[S_\tau(z)]$ lies between $0$ and $\theta$, its distance from $\theta$ is nondecreasing in $\tau$, $\operatorname{var}(S_\tau(z))$ is nonincreasing in $\tau$, and both the mean and the variance tend to $0$ as $\tau\to\infty$. Under A1–A5 with $\mathbf{X}'\mathbf{X}=\mathbf{I}$, $\hat\beta_j\sim N(\beta_j,\sigma^2)$, so the lasso coefficients of Theorem 6.22 have squared bias nondecreasing and variance nonincreasing in $\lambda$.

**Proof.** **Step 1: closed forms.** By symmetry take $\theta\geqslant0$. Write $z=\theta+\sigma Z$ and, with Unit 5's lemma **truncated normal moments**, $g(a)=\operatorname{\mathsf{E}}[(Z-a)^+]$, so $g'(a)=-\{1-\Phi(a)\}$. Integrating by parts as in that lemma, $\operatorname{\mathsf{E}}[\{(Z-a)^+\}^2]=\int_a^\infty(t-a)^2\varphi(t)\,\mathrm{d} t=(1+a^2)\{1-\Phi(a)\}-a\varphi(a)=:k(a)$, with $k'(a)=2a\{1-\Phi(a)\}-2\varphi(a)=-2g(a)$. Since $S_\tau(z)=(z-\tau)^+-(-z-\tau)^+$, the two parts vanishing together or one at a time, and $-z=-\theta+\sigma Z'$ with $Z'=-Z$ standard normal,

$$
\mu(\tau)=\operatorname{\mathsf{E}} S_\tau(z)=\sigma g(a_-)-\sigma g(a_+),\qquad
\operatorname{\mathsf{E}} S_\tau(z)^2=\sigma^2k(a_-)+\sigma^2k(a_+),\qquad a_\mp=\frac{\tau\mp\theta}{\sigma}.
$$

**Step 2: the mean.** $\mu'(\tau)=\Phi(a_-)-\Phi(a_+)\leqslant0$, as $a_-\leqslant a_+$, so $\mu$ is nonincreasing by the prerequisites' mean value theorem. $\mu(0)=\operatorname{\mathsf{E}} z=\theta$, since $S_0(z)=z$, and $\mu(\tau)\to0$ because $g(a)\to0$ as $a\to\infty$ (dominated convergence); a nonincreasing function with these limits lies in $[0,\theta]$, so $\theta-\mu(\tau)$ is nonnegative and nondecreasing.

**Step 3: the variance.** $v(\tau)=\operatorname{\mathsf{E}} S_\tau(z)^2-\mu(\tau)^2$ has

$$
v'(\tau)=-2\sigma g(a_-)-2\sigma g(a_+)+2\mu(\tau)\{\Phi(a_+)-\Phi(a_-)\}
\leqslant-2\sigma\{g(a_-)+g(a_+)\}+2\mu(\tau)\leqslant0 ,
$$

using $0\leqslant\Phi(a_+)-\Phi(a_-)\leqslant1$, $\mu(\tau)\geqslant0$, and $\mu(\tau)=\sigma g(a_-)-\sigma g(a_+)\leqslant\sigma g(a_-)+\sigma g(a_+)$. By the prerequisites' mean value theorem $v$ is nonincreasing, and $v(\tau)\leqslant\operatorname{\mathsf{E}} S_\tau(z)^2\to0$.

**Step 4: the design.** Under A1–A5, $\hat\beta_j=\mathbf{x}_j'\mathbf{Y}_c=\mathbf{x}_j'\mathbf{Y}$ is a linear function of the normal $\mathbf{Y}$ with mean $\mathbf{x}_j'\mathbf{X}\boldsymbol{\beta}=\beta_j$ and variance $\sigma^2\mathbf{x}_j'\mathbf{x}_j=\sigma^2$, and $\tau=\lambda/2$. ∎

**Example 6.3** (Constraint regions and shrinkage; ISLR §6.2.2, Figures 6.7 and 6.10). Two correlated predictors; the lasso at $\lambda=40$, with $s=\|\hat{\boldsymbol{\beta}}\|_1$, and the ridge estimate with $\|\hat{\boldsymbol{\beta}}_\lambda\|=s$, whose disc passes through the corners of the diamond $\|\boldsymbol{\beta}\|_1\leqslant s$; and the two shrinkage rules of Theorem 6.22 at $\lambda=1$:

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 2.9)
set.seed(4); nn <- 60; x1 <- rnorm(nn); x2 <- 0.6 * x1 + sqrt(1 - 0.36) * rnorm(nn)
Xm <- scale(cbind(x1, x2), TRUE, FALSE); yv <- drop(x1 + rnorm(nn)); yv <- yv - mean(yv)
A2 <- crossprod(Xm); bh <- drop(solve(A2, crossprod(Xm, yv)))
soft <- function(a, t) sign(a) * pmax(abs(a) - t, 0)
bl <- c(0, 0); for (it in 1:3000) for (j in 1:2) {
  r <- yv - Xm[, -j] * bl[-j]; bl[j] <- soft(sum(Xm[, j] * r), 20) / sum(Xm[, j]^2) }
s1 <- sum(abs(bl))
lr <- uniroot(function(l) sqrt(sum(solve(A2 + l * diag(2), crossprod(Xm, yv))^2)) - s1, c(1e-8, 1e5))$root
br <- drop(solve(A2 + lr * diag(2), crossprod(Xm, yv)))
round(rbind(least_squares = bh, lasso = bl, ridge = br), 4)
qf <- function(b) drop(t(b - bh) %*% A2 %*% (b - bh)); g2 <- seq(-0.85, 1.75, length.out = 250)
Zc <- outer(g2, g2, Vectorize(function(a, b) qf(c(a, b)))); th <- seq(0, 2 * pi, length.out = 300)
op <- par(mfrow = c(1, 4), mar = c(3.2, 3.2, 1.6, 0.4), mgp = c(1.9, 0.6, 0), cex = 0.7)
for (w in 1:2) { bs <- if (w == 1) bl else br
  contour(g2, g2, Zc, levels = qf(bs) * c(0.2, 1, 2.5), drawlabels = FALSE, col = "grey55", asp = 1,
          xlab = expression(beta[1]), ylab = expression(beta[2]), main = c("lasso", "ridge")[w])
  if (w == 1) polygon(c(s1, 0, -s1, 0), c(0, s1, 0, -s1), border = 4, col = adjustcolor(4, 0.15))
  else polygon(s1 * cos(th), s1 * sin(th), border = 4, col = adjustcolor(4, 0.15))
  points(bs[1], bs[2], pch = 19, col = 2); points(bh[1], bh[2], pch = 19) }
a <- seq(-3, 3, length.out = 301)
plot(a, a / 2, type = "l", col = 4, lwd = 2, asp = 1, xlab = "least squares", ylab = "estimate", main = "ridge")
abline(0, 1, lty = 3)
plot(a, soft(a, 0.5), type = "l", col = 2, lwd = 2, asp = 1, xlab = "least squares", ylab = "", main = "lasso")
abline(0, 1, lty = 3)
par(op)

*Figure 6.3.* Left pair: contours of $\mathrm{RSS}$ about the least squares estimate (black) and the constraint regions $|\beta_1|+|\beta_2|\leqslant s$ and $\beta_1^2+\beta_2^2\leqslant s^2$, with the constrained estimates (red) (ISLR Figure 6.7). Right pair: ridge and lasso against least squares for an orthonormal design at $\lambda=1$, the dotted line being the identity (ISLR Figure 6.10).

The lasso estimate $(0.6493,0)$ lies at a corner of the diamond, with $\hat\beta_2=0$, while the ridge estimate with the same budget, $(0.6353,0.1342)$, keeps both coordinates; least squares is $(1.0860,-0.0510)$.

### 2.4 The Bayesian interpretation

**Proposition 6.24** (Posterior modes and means; ISLR §6.2.2). Let $\mathbf{Y}\mid\boldsymbol{\beta}\sim N(\mathbf{X}\boldsymbol{\beta},\sigma^2\mathbf{I}_n)$ with $\mathbf{X}$ fixed, no intercept and $\sigma^2$ known, and let $\beta_1,\ldots,\beta_p$ be a priori independent with density $\pi_0$. The posterior density $\pi(\boldsymbol{\beta}\mid\mathbf{Y})=f(\mathbf{Y}\mid\boldsymbol{\beta})\prod_j\pi_0(\beta_j)/q(\mathbf{Y})$, with $q(\mathbf{Y})=\int f(\mathbf{Y}\mid\boldsymbol{\beta})\prod_j\pi_0(\beta_j)\,\mathrm{d}\boldsymbol{\beta}\in(0,\infty)$, satisfies:

(i) for the double-exponential prior $\pi_0(t)=e^{-|t|/\omega}/(2\omega)$, the posterior modes are the minimisers of $\|\mathbf{Y}-\mathbf{X}\boldsymbol{\beta}\|^2+\lambda\|\boldsymbol{\beta}\|_1$ with $\lambda=2\sigma^2/\omega$, the lasso without intercept;

(ii) for the normal prior $\pi_0(t)=e^{-t^2/(2\nu)}/\sqrt{2\pi \nu}$, the posterior mode and the posterior mean are both the ridge estimate without intercept, $(\mathbf{X}'\mathbf{X}+\lambda\mathbf{I})^{-1}\mathbf{X}'\mathbf{Y}$ with $\lambda=\sigma^2/\nu$.

**Proof.** **Step 1: the posterior.** The integrand defining $q(\mathbf{Y})$ is positive, and at most $(2\pi\sigma^2)^{-n/2}\prod_j\pi_0(\beta_j)$, whose integral is $(2\pi\sigma^2)^{-n/2}$ by Fubini–Tonelli; so $0<q(\mathbf{Y})<\infty$ and $\pi$ is a density. Since $f(\mathbf{Y}\mid\boldsymbol{\beta})=(2\pi\sigma^2)^{-n/2}\exp\{-\|\mathbf{Y}-\mathbf{X}\boldsymbol{\beta}\|^2/(2\sigma^2)\}$, $-2\sigma^2\ln\pi(\boldsymbol{\beta}\mid\mathbf{Y})=\|\mathbf{Y}-\mathbf{X}\boldsymbol{\beta}\|^2-2\sigma^2\sum_j\ln \pi_0(\beta_j)+C$, with $C$ free of $\boldsymbol{\beta}$; the modes are the minimisers of this.

**Step 2: (i).** $-2\sigma^2\ln \pi_0(t)=(2\sigma^2/\omega)|t|+2\sigma^2\ln(2\omega)$, so the modes minimise $\|\mathbf{Y}-\mathbf{X}\boldsymbol{\beta}\|^2+(2\sigma^2/\omega)\|\boldsymbol{\beta}\|_1$; they exist by the proof of Theorem 6.15 with $\mathbf{Y}$ in place of $\mathbf{Y}_c$.

**Step 3: (ii).** Now the modes minimise $\|\mathbf{Y}-\mathbf{X}\boldsymbol{\beta}\|^2+(\sigma^2/\nu)\|\boldsymbol{\beta}\|^2$, whose unique minimiser is $\mathbf{m}=\mathbf{A}^{-1}\mathbf{X}'\mathbf{Y}$ with $\mathbf{A}=\mathbf{X}'\mathbf{X}+(\sigma^2/\nu)\mathbf{I}$, as in the proof of Theorem 6.3; completing the square there, $\pi(\boldsymbol{\beta}\mid\mathbf{Y})=K\exp\{-(\boldsymbol{\beta}-\mathbf{m})'\mathbf{A}(\boldsymbol{\beta}-\mathbf{m})/(2\sigma^2)\}$ for a constant $K$. With $\mathbf{u}=\boldsymbol{\beta}-\mathbf{m}$, the function $\mathbf{u}\exp\{-\mathbf{u}'\mathbf{A}\mathbf{u}/(2\sigma^2)\}$ is odd and integrable, as $\mathbf{u}'\mathbf{A}\mathbf{u}\geqslant(\sigma^2/\nu)\|\mathbf{u}\|^2$, so its integral is $\mathbf{0}$ (substitute $\mathbf{u}\mapsto-\mathbf{u}$) and the posterior mean is $\mathbf{m}$. ∎

**Remark** (ISLR §6.2.2). The double-exponential density has a corner at zero, where the normal density is flat. The corner is the non-differentiability of $|\beta_j|$ at $0$, which gives Theorem 6.18 an inequality rather than an equation at $\hat\beta_j=0$: the posterior mode under the double-exponential prior can have coordinates exactly zero, under the normal prior only by coincidence (the remark after Theorem 6.20).

### 2.5 The lasso on the Credit data and in simulation

**Example 6.4** (Lasso coefficient paths; ISLR §6.2.2, Figure 6.6). The lasso path of the standardised `Credit` predictors, computed by `glmnet` without its own standardisation and intercept; it minimises $\mathrm{RSS}/(2n)+\lambda'\|\boldsymbol{\beta}\|_1$, so $\lambda'=\lambda/(2n)$. The entry points are read on the grid:

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 2.5)
yc_c <- yc - mean(yc)
lmax_c <- 2 * max(abs(crossprod(Xs_c, yc_c)))
gl_c <- glmnet(Xs_c, yc_c, alpha = 1, standardize = FALSE, intercept = FALSE,
               lambda = lmax_c * 10^seq(0, -4, length = 400) / (2 * n_c), thresh = 1e-12, maxit = 1e6)
BL_c <- as.matrix(coef(gl_c))[-1, ]; lam_l <- 2 * n_c * gl_c$lambda
entry <- apply(BL_c, 1, function(b) { w <- which(b != 0); if (length(w)) lam_l[w[1]] else NA })
c(lambda_max = lmax_c); round(sort(entry, decreasing = TRUE))
ratio_l <- colSums(abs(BL_c)) / sum(abs(b_ls))
op <- par(mfrow = c(1, 2), mar = c(3.2, 3.8, 0.6, 0.6), mgp = c(2.1, 0.6, 0))
matplot(lam_l, t(BL_c), type = "l", lty = 1, log = "x", col = col_c, xlab = "lambda", ylab = "standardised coefficient")
legend("topright", c("Income", "Limit", "Rating", "StudentYes"), col = c(1, 2, 4, 3), lty = 1, bty = "n", cex = 0.7)
matplot(ratio_l, t(BL_c), type = "l", lty = 1, col = col_c, xlab = "l1 norm ratio", ylab = "")
par(op)

*Figure 6.4.* Standardised lasso coefficients for `Credit` against $\lambda$ (log scale) and against $\|\hat{\boldsymbol{\beta}}_\lambda\|_1/\|\hat{\boldsymbol{\beta}}\|_1$ (ISLR Figure 6.6).

`Rating` enters first, at $\lambda_{\max}=317{,}250$ (Corollary 6.19), then `StudentYes` ($95{,}521$) and `Limit` ($91{,}211$) almost together, and `Income` ($45{,}635$), the order ISLR reports; the other seven enter at $16{,}150$ or below.

**Example 6.5** (Lasso against ridge; ISLR §6.2.2, Figures 6.8 and 6.9). The design of Example 6.2, with the dense $\boldsymbol{\beta}$ and with a sparse one, $\beta_1=\beta_2=2.5$ and the other $43$ coefficients zero. The lasso's squared bias and variance are estimated from $200$ independent responses: with $\bar{\boldsymbol{\beta}}$ the average of the $200$ estimates and $\hat{\boldsymbol{\Sigma}}$ their sample variance matrix, $\|\bar{\boldsymbol{\beta}}-\boldsymbol{\beta}\|^2-\operatorname{tr}\hat{\boldsymbol{\Sigma}}/200$ is unbiased for $\|\operatorname{\mathsf{E}}\hat{\boldsymbol{\beta}}-\boldsymbol{\beta}\|^2$, because $\operatorname{\mathsf{E}}\|\bar{\boldsymbol{\beta}}-\boldsymbol{\beta}\|^2=\|\operatorname{\mathsf{E}}\hat{\boldsymbol{\beta}}-\boldsymbol{\beta}\|^2+\operatorname{tr}\operatorname{var}(\hat{\boldsymbol{\beta}})/200$ and $\operatorname{\mathsf{E}}\hat{\boldsymbol{\Sigma}}=\operatorname{var}(\hat{\boldsymbol{\beta}})$. Ridge's come from Proposition 6.9. Both are plotted against their average training $R^2$:

In [ ]:
sim_both <- function(beta, R = 200) {
  mu <- drop(X_s %*% beta); sv_s <- svd(X_s)
  arr <- replicate(R, { yy <- mu + rnorm(n_s)
    f <- glmnet(X_s, yy, alpha = 1, lambda = lam_s / (2 * n_s), standardize = FALSE, intercept = FALSE, thresh = 1e-12)
    bm <- as.matrix(coef(f))[-1, ]
    r2_l <- 1 - colSums((yy - X_s %*% bm)^2) / sum(yy^2)
    r2_r <- sapply(lam_s, function(l) { b <- sv_s$v %*% (sv_s$d / (sv_s$d^2 + l) * crossprod(sv_s$u, yy))
      1 - sum((yy - X_s %*% b)^2) / sum(yy^2) })
    rbind(bm, r2_l, r2_r) })
  bmean <- apply(arr[1:p_s, , ], c(1, 2), mean); bvar <- apply(arr[1:p_s, , ], c(1, 2), var)
  cbind(bias2 = colSums((bmean - beta)^2) - colSums(bvar) / R, var = colSums(bvar),
        R2_lasso = rowMeans(arr[p_s + 1, , ]), R2_ridge = rowMeans(arr[p_s + 2, , ])) }
set.seed(689); ls_d <- sim_both(b_dense); ls_s <- sim_both(b_sparse)
re_s <- ridge_exact(b_sparse)
ends <- function(m) c(lambda_1000 = m[1], minimum = min(m), lambda_0.01 = m[length(m)])
round(rbind(ridge_dense = ends(1 + rowSums(re_d)), lasso_dense = ends(1 + ls_d[, 1] + ls_d[, 2]),
            ridge_sparse = ends(1 + rowSums(re_s)), lasso_sparse = ends(1 + ls_s[, 1] + ls_s[, 2])), 3)
op <- par(mfrow = c(1, 2), mar = c(3.2, 3.8, 1.6, 0.6), mgp = c(2.1, 0.6, 0))
for (w in 1:2) { L <- list(ls_d, ls_s)[[w]]; Rr <- list(re_d, re_s)[[w]]
  matplot(L[, "R2_lasso"], cbind(L[, 1], L[, 2], 1 + L[, 1] + L[, 2]), type = "l", lty = 1, col = c(1, 3, 6),
          xlab = "training R-squared", ylab = "", ylim = c(0, 16), main = c("dense", "sparse")[w])
  matlines(L[, "R2_ridge"], cbind(Rr, 1 + rowSums(Rr)), lty = 3, col = c(1, 3, 6)) }
par(op)

*Figure 6.5.* Squared bias (black), variance (green) and expected test error (purple) of the lasso (solid) and ridge (dotted) against training $R^2$, for the dense and the sparse $\boldsymbol{\beta}$ (ISLR Figures 6.8 and 6.9).

Each method is best at an intermediate $\lambda$, its minimum lying below its values at both ends of the grid. With all $45$ coefficients equal, ridge reaches an expected test error of $3.376$ and the lasso $5.524$; with two nonzero coefficients, the lasso reaches $1.202$ and ridge $4.146$. For ridge the variance falls and the squared bias rises with $\lambda$ by Theorem 6.10; Proposition 6.23 proves the same for the lasso only for orthonormal designs.

**Remark** (ISLR §6.2.2). Neither method dominates. The lasso is expected to do better when a few predictors have substantial coefficients and the rest are zero or small, ridge when the response depends on many predictors with coefficients of similar size, as in Example 6.5. The number of relevant predictors is unknown in practice, and cross-validation chooses between them.

### 2.6 Coordinate descent and the elastic net (beyond the syllabus)

> **Reading.** Beyond the syllabus: ISLR §6.2.2 mentions efficient algorithms for the whole path without describing them; the elastic net appears only through the `alpha` argument of `glmnet` in the lab.

**Proposition 6.25** (The coordinate update). Fix all coordinates but the $j$th, let $\mathbf{r}^{(j)}=\mathbf{Y}_c-\sum_{l\neq j}\mathbf{x}_l\beta_l$, and let $\mathbf{x}_j\neq\mathbf{0}$. The minimiser of $\beta_j\mapsto L_\lambda(\boldsymbol{\beta})$ is $S_{\lambda/2}(\mathbf{x}_j'\mathbf{r}^{(j)})/\|\mathbf{x}_j\|^2$.

**Proof.** Up to terms free of $\beta_j$, $L_\lambda$ is $\|\mathbf{r}^{(j)}-\mathbf{x}_j\beta_j\|^2+\lambda|\beta_j| =\|\mathbf{x}_j\|^2\{(\beta_j-a)^2+2\tau|\beta_j|\}+\text{const}$, with $a=\mathbf{x}_j'\mathbf{r}^{(j)}/\|\mathbf{x}_j\|^2$ and $\tau=\lambda/(2\|\mathbf{x}_j\|^2)$. By Lemma 6.21 the minimiser is $\operatorname{sign}(a)(|a|-\tau)_+$, which equals $S_{\lambda/2}(\mathbf{x}_j'\mathbf{r}^{(j)})/\|\mathbf{x}_j\|^2$. ∎

**Theorem 6.26** (Coordinatewise optimality is optimality). If every column of $\mathbf{X}$ is nonzero and $\boldsymbol{\beta}$ satisfies the update of Proposition 6.25 at every $j$, then $\boldsymbol{\beta}$ minimises $L_\lambda$.

**Proof.** With $z=\mathbf{x}_j'\mathbf{r}^{(j)}$, $c_j=\mathbf{x}_j'(\mathbf{Y}_c-\mathbf{X}\boldsymbol{\beta})=z-\|\mathbf{x}_j\|^2\beta_j$. If $\beta_j=0$, the update forces $|z|\leqslant\lambda/2$, so $|2c_j|\leqslant\lambda$. If $\beta_j\neq0$, then $\|\mathbf{x}_j\|^2\beta_j=\operatorname{sign}(z)(|z|-\lambda/2)$ with $\operatorname{sign}(\beta_j)=\operatorname{sign}(z)$, so $c_j=\operatorname{sign}(\beta_j)\lambda/2$. Theorem 6.18 applies. ∎

**Remark**. Sweeping $j=1,\ldots,p$ with the update until nothing changes is coordinate descent, and a converged sweep certifies a minimum by Theorem 6.26. `glmnet` runs it on a decreasing grid of $\lambda$ that starts at $\lambda_{\max}$ of Corollary 6.19, each fit starting from the previous one.

**Definition 6.27** (The elastic net). For $\lambda_1,\lambda_2\geqslant0$, the **elastic net** minimises $E(\boldsymbol{\beta})=\|\mathbf{Y}_c-\mathbf{X}\boldsymbol{\beta}\|^2+\lambda_1\|\boldsymbol{\beta}\|_1+\lambda_2\|\boldsymbol{\beta}\|^2$; `glmnet` indexes the family by $\alpha\in[0,1]$, with $\alpha=1$ the lasso and $\alpha=0$ ridge.

**Theorem 6.28** (Uniqueness and grouping). (i) If $\lambda_2>0$, $E$ has exactly one minimiser.

(ii) Let $\mathbf{x}_j=\mathbf{x}_l$ for some $j\neq l$. If $\lambda_2>0$ the minimiser has $\hat\beta_j=\hat\beta_l$. If $\lambda_2=0<\lambda_1$ and $\hat{\boldsymbol{\beta}}$ is a lasso estimator, replacing $(\hat\beta_j,\hat\beta_l)$ by $(t,\hat\beta_j+\hat\beta_l-t)$, for any $t$ between $0$ and $\hat\beta_j+\hat\beta_l$, gives another lasso estimator.

**Proof.** (i) The first two terms are convex and $\lambda_2\|\boldsymbol{\beta}\|^2$ is strictly convex, so $E$ is strictly convex, and $E(\boldsymbol{\beta})\geqslant\lambda_2\|\boldsymbol{\beta}\|^2$ makes it coercive; apply the prerequisites' lemma **coercive minimisation**.

(ii) Exchanging coordinates $j$ and $l$ leaves $\mathbf{X}\boldsymbol{\beta}$ and both penalties unchanged, so it maps minimisers to minimisers; with one minimiser, it is fixed, $\hat\beta_j=\hat\beta_l$. For the lasso, $\mathbf{x}_j=\mathbf{x}_l$ gives $c_j=c_l$ in Theorem 6.18, so $\hat\beta_j,\hat\beta_l$ do not have opposite signs; with $\kappa=\hat\beta_j+\hat\beta_l$, $|\hat\beta_j|+|\hat\beta_l|=|\kappa|=|t|+|\kappa-t|$ for $t$ between $0$ and $\kappa$, and the fit is unchanged, so the criterion is unchanged. ∎

**Remark**. Two identical predictors share their weight equally under the elastic net, while the lasso may split it in any proportion of one sign.

## 3 Selecting the Tuning Parameter

> **Reading.** ISLR §6.2.3. SRM learning outcome 2(g).

**Definition 6.29** (Cross-validation for $\lambda$; ISLR §6.2.3). Fix a grid $\lambda_1>\cdots>\lambda_G$ and a partition of the observations into folds $F_1,\ldots,F_V$. Let $\hat f^{(-v)}_\lambda$ be the fit at $\lambda$ to the data outside $F_v$, $e_v(\lambda)=|F_v|^{-1}\sum_{i\in F_v}\{y_i-\hat f^{(-v)}_\lambda(\mathbf{x}_i)\}^2$ the error on fold $v$, and

$$
\mathrm{CV}(\lambda)=\sum_{v=1}^V\frac{|F_v|}{n}e_v(\lambda),\qquad
\mathrm{SE}(\lambda)=\Bigl\{\frac{1}{V-1}\sum_{v=1}^V\frac{|F_v|}{n}\bigl(e_v(\lambda)-\mathrm{CV}(\lambda)\bigr)^2\Bigr\}^{1/2} .
$$

$\lambda_{\min}$ minimises $\mathrm{CV}$ over the grid, and the one-standard-error rule picks the largest $\lambda_{1\mathrm{SE}}$ with $\mathrm{CV}(\lambda_{1\mathrm{SE}})\leqslant\mathrm{CV}(\lambda_{\min})+\mathrm{SE}(\lambda_{\min})$, the most penalised fit within one standard error of the best. The model is then refitted to all the data at the selected $\lambda$. With folds of equal size, $\mathrm{CV}$ is Unit 1's $\mathrm{CV}_{(k)}$ (definition **$k$-fold CV**) and $\mathrm{SE}$ is the $\mathrm{SE}_d$ of Unit 5's definition **one-standard-error rule**; the weights $|F_v|/n$ are those of `cv.glmnet`. With $V=n$ this is leave-one-out cross-validation (LOOCV).

**Proposition 6.30** (LOOCV for ridge costs one fit; ISLR §6.2.3). Let $\tilde{\mathbf{X}}=[\mathbf{1}\ \mathbf{X}]$ with $\mathbf{X}$ centred and fixed, $n\geqslant2$, $\lambda>0$, $\boldsymbol{\Omega}=\operatorname{diag}(0,\lambda,\ldots,\lambda)$, $\mathbf{A}=\tilde{\mathbf{X}}'\tilde{\mathbf{X}}+\boldsymbol{\Omega}$ and $\mathbf{S}=\tilde{\mathbf{X}}\mathbf{A}^{-1}\tilde{\mathbf{X}}'$; as $\tilde{\mathbf{X}}'\tilde{\mathbf{X}}=\operatorname{diag}(n,\mathbf{X}'\mathbf{X})$ for centred $\mathbf{X}$, $\mathbf{A}=\operatorname{diag}(n,\mathbf{X}'\mathbf{X}+\lambda\mathbf{I})$, so that $\hat{\mathbf{Y}}=\mathbf{S}\mathbf{Y}$ is the ridge fit of Proposition 6.12. Let $\hat y_{(i)}$ be the prediction at $\tilde{\mathbf{x}}_i$ from $\hat{\boldsymbol{\beta}}_{(i)}=(\tilde{\mathbf{X}}_{(i)}'\tilde{\mathbf{X}}_{(i)}+\boldsymbol{\Omega})^{-1}\tilde{\mathbf{X}}_{(i)}'\mathbf{Y}_{(i)}$, the same fit without observation $i$ and with the same $\boldsymbol{\Omega}$. Then $S_{ii}<1$ and

$$
y_i-\hat y_{(i)}=\frac{y_i-\hat y_i}{1-S_{ii}},\qquad
\mathrm{CV}_{(n)}(\lambda)=\frac1n\sum_{i=1}^n\Bigl(\frac{y_i-\hat y_i}{1-S_{ii}}\Bigr)^2 .
$$

**Proof.** **Step 1: invertibility.** Write $\mathbf{A}_{(i)}=\mathbf{A}-\tilde{\mathbf{x}}_i\tilde{\mathbf{x}}_i'=\tilde{\mathbf{X}}_{(i)}'\tilde{\mathbf{X}}_{(i)}+\boldsymbol{\Omega}$. For $\mathbf{v}=(v_0,\mathbf{w}')'\neq\mathbf{0}$, $\mathbf{v}'\mathbf{A}_{(i)}\mathbf{v}=\|\tilde{\mathbf{X}}_{(i)}\mathbf{v}\|^2+\lambda\|\mathbf{w}\|^2$, positive if $\mathbf{w}\neq\mathbf{0}$, and equal to $(n-1)v_0^2>0$ if $\mathbf{w}=\mathbf{0}$. So $\mathbf{A}_{(i)}$, and likewise $\mathbf{A}$, is positive definite. The prerequisites' lemma **Sherman–Morrison**, applied to $\mathbf{A}=\mathbf{A}_{(i)}-\tilde{\mathbf{x}}_i(-\tilde{\mathbf{x}}_i)'$, gives $S_{ii}=\tilde{\mathbf{x}}_i'\mathbf{A}^{-1}\tilde{\mathbf{x}}_i=h/(1+h)<1$ with $h=\tilde{\mathbf{x}}_i'\mathbf{A}_{(i)}^{-1}\tilde{\mathbf{x}}_i\geqslant0$.

**Step 2: the deleted fit.** The same lemma, applied to $\mathbf{A}_{(i)}=\mathbf{A}-\tilde{\mathbf{x}}_i\tilde{\mathbf{x}}_i'$ with $1-S_{ii}\neq0$, gives $\mathbf{A}_{(i)}^{-1}=\mathbf{A}^{-1}+\mathbf{A}^{-1}\tilde{\mathbf{x}}_i\tilde{\mathbf{x}}_i'\mathbf{A}^{-1}/(1-S_{ii})$. With $\tilde{\mathbf{X}}_{(i)}'\mathbf{Y}_{(i)}=\tilde{\mathbf{X}}'\mathbf{Y}-\tilde{\mathbf{x}}_iy_i$ and $\tilde{\mathbf{x}}_i'\mathbf{A}^{-1}\tilde{\mathbf{X}}'\mathbf{Y}=\hat y_i$,

$$
\hat y_{(i)}=\tilde{\mathbf{x}}_i'\mathbf{A}_{(i)}^{-1}(\tilde{\mathbf{X}}'\mathbf{Y}-\tilde{\mathbf{x}}_iy_i)
=(\hat y_i-S_{ii}y_i)\Bigl(1+\frac{S_{ii}}{1-S_{ii}}\Bigr)=\frac{\hat y_i-S_{ii}y_i}{1-S_{ii}},
$$

and $y_i-\hat y_{(i)}=\{(1-S_{ii})y_i-\hat y_i+S_{ii}y_i\}/(1-S_{ii})$ is the first identity. Averaging the squares over $i$ gives the second. ∎

**Remark**. This is Unit 1's proposition **LOOCV shortcut for least squares** with the penalty $\boldsymbol{\Omega}$ added. The standardisation is computed once, on all the data; a deleted fit that re-standardises changes the design, which Proposition 6.30 does not cover.

**Remark**. The minimum of the curve is optimistic as an estimate of the error at the selected $\lambda$: for every grid point $\lambda_0$, $\min_\lambda\mathrm{CV}(\lambda)\leqslant\mathrm{CV}(\lambda_0)$, so $\operatorname{\mathsf{E}}[\min_\lambda\mathrm{CV}(\lambda)]\leqslant\min_{\lambda_0}\operatorname{\mathsf{E}}[\mathrm{CV}(\lambda_0)]$. To estimate the error of the whole procedure, the choice of $\lambda$ is repeated inside each fold of an outer cross-validation.

**Example 6.6** (LOOCV for ridge on the Credit data; ISLR §6.2.3, Figure 6.12). Proposition 6.30 on the standardised predictors of Example 6.1, against $400$ refits at the selected $\lambda$:

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 2.4)
loo_c <- function(l) { S <- sv_c$u %*% (sv_c$d^2 / (sv_c$d^2 + l) * t(sv_c$u)) + matrix(1 / n_c, n_c, n_c)
  mean(((yc - drop(S %*% yc)) / (1 - diag(S)))^2) }
lg_c <- 10^seq(-2, 4, length = 121); cv_loo <- sapply(lg_c, loo_c); lb_c <- lg_c[which.min(cv_loo)]
brute_c <- mean(sapply(1:n_c, function(i) { Xi <- Xs_c[-i, ]; yi <- yc[-i]; mi <- colMeans(Xi)
  b <- solve(crossprod(sweep(Xi, 2, mi)) + lb_c * diag(p_c), crossprod(sweep(Xi, 2, mi), yi - mean(yi)))
  (yc[i] - mean(yi) - sum((Xs_c[i, ] - mi) * b))^2 }))
round(c(lambda_min = lb_c, norm_ratio = sqrt(sum(ridge_c(lb_c)^2) / sum(b_ls^2)), LOOCV_min = min(cv_loo),
  LOOCV_at_0.01 = cv_loo[1], LOOCV_at_10000 = cv_loo[121]), 3)
signif(c(refits_minus_shortcut = brute_c - loo_c(lb_c)), 3)
op <- par(mfrow = c(1, 2), mar = c(3.2, 3.8, 0.6, 0.6), mgp = c(2.1, 0.6, 0))
plot(lg_c, cv_loo, type = "l", log = "x", xlab = "lambda", ylab = "LOOCV error", xaxt = "n"); ax10(); abline(v = lb_c, lty = 2)
matplot(lam_c, t(B_c), type = "l", lty = 1, log = "x", col = col_c, xlab = "lambda", ylab = "standardised coefficient", xaxt = "n")
ax10(); abline(v = lb_c, lty = 2)
par(op)

*Figure 6.6.* Leave-one-out cross-validation error of ridge regression on `Credit`, and the coefficient paths, with the selected $\lambda$ dashed (ISLR Figure 6.12).

Leave-one-out cross-validation is smallest at $\lambda=0.355$, where the norm ratio is $0.970$: a small amount of shrinkage. The curve is nearly flat below it ($10{,}064.630$ at the minimum, $10{,}072.093$ at $\lambda=0.01$), as ISLR observes; the shortcut agrees with the $400$ refits to rounding error.

**Example 6.7** (Ten-fold cross-validation for the lasso on sparse data; ISLR §6.2.3, Figure 6.13). One training set from the sparse model of Example 6.5:

In [ ]:
set.seed(613); y_sp <- drop(X_s %*% b_sparse + rnorm(n_s))
cv_sp <- cv.glmnet(X_s, y_sp, alpha = 1, standardize = FALSE, intercept = FALSE)
cf_min <- as.matrix(coef(cv_sp, s = "lambda.min"))[-1, 1]; cf_1se <- as.matrix(coef(cv_sp, s = "lambda.1se"))[-1, 1]
b_ls_sp <- drop(solve(XtX_s, crossprod(X_s, y_sp)))
summ <- function(b) c(beta1 = b[[1]], beta2 = b[[2]], noise_nonzero = sum(b[-(1:2)] != 0), max_abs_noise = max(abs(b[-(1:2)])))
rbind(lambda_min = summ(cf_min), lambda_1se = summ(cf_1se), least_squares = summ(b_ls_sp))
fit_sp <- glmnet(X_s, y_sp, alpha = 1, standardize = FALSE, intercept = FALSE)
op <- par(mfrow = c(1, 2), mar = c(3.2, 3.8, 1.6, 0.6), mgp = c(2.1, 0.6, 0))
plot(cv_sp); plot(fit_sp, xvar = "lambda", col = c(2, 4, rep("grey60", p_s - 2)))
abline(v = -log(cv_sp$lambda.min), lty = 2)
par(op)

*Figure 6.7.* Ten-fold cross-validation error of the lasso on one sparse training set, with $\lambda_{\min}$ and $\lambda_{1\mathrm{SE}}$ dotted, and the coefficient paths against $-\ln\lambda'$, with $\lambda_{\min}$ dashed and the two signal coefficients in colour (ISLR Figure 6.13).

At $\lambda_{\min}$ the two signal coefficients are $2.089$ and $2.131$, and $19$ noise predictors are kept, none above $0.3417$ in absolute value; at $\lambda_{1\mathrm{SE}}$, $8$ are kept, none above $0.2129$; least squares gives the signals $1.767$ and $2.264$ and the noise up to $1.0119$. On this training set cross-validation does not isolate the two signals, as it does in ISLR's figure, but it shrinks every noise coefficient far below them.

## 4 Nearest Neighbour Regression

> **Reading.** ISLR §3.5. SRM learning outcome 2(g).

### 4.1 KNN regression and its error

**Definition 6.31** (KNN regression; ISLR §3.5). Given training pairs $(\mathbf{x}_1,y_1),\ldots,(\mathbf{x}_n,y_n)$ with $\mathbf{x}_i\in\mathbb{R}^p$, an integer $1\leqslant K\leqslant n$ and a point $\mathbf{x}_0$, let $\mathcal{N}_0$ be the set of the $K$ training inputs nearest $\mathbf{x}_0$ in Euclidean distance, ties broken by a fixed rule. The **KNN regression** estimate is $\hat f(\mathbf{x}_0)=K^{-1}\sum_{i\in\mathcal{N}_0}y_i$, the regression counterpart of Unit 1's KNN classifier, which averages the indicators $I(y_i=j)$ instead. Linear regression is **parametric**, KNN **non-parametric**.

**Theorem 6.32** (The error of KNN at a point; ISLR §3.5). Let the inputs be fixed, $y_i=f(\mathbf{x}_i)+\varepsilon_i$ with the $\varepsilon_i$ uncorrelated, of mean $0$ and variance $\sigma^2$, and $y_0=f(\mathbf{x}_0)+\varepsilon_0$ a new observation with $\varepsilon_0$ uncorrelated with the training errors, of mean $0$ and variance $\sigma^2$. Then

$$
\operatorname{\mathsf{E}}\bigl(y_0-\hat f(\mathbf{x}_0)\bigr)^2=\sigma^2
+\Bigl(f(\mathbf{x}_0)-\frac1K\sum_{i\in\mathcal{N}_0}f(\mathbf{x}_i)\Bigr)^2+\frac{\sigma^2}{K} .
$$

**Proof.** With fixed inputs $\mathcal{N}_0$ is a fixed set, and $\hat f(\mathbf{x}_0)$ is the average of $K$ uncorrelated responses, with mean $\bar f_0=K^{-1}\sum_{i\in\mathcal{N}_0}f(\mathbf{x}_i)$ and variance $\sigma^2/K$. Expanding $y_0-\hat f(\mathbf{x}_0)=\varepsilon_0+\{f(\mathbf{x}_0)-\bar f_0\}-\{\hat f(\mathbf{x}_0)-\bar f_0\}$, the three cross products have mean zero: $\operatorname{\mathsf{E}}[\varepsilon_0]=0$, $\operatorname{\mathsf{E}}[\hat f(\mathbf{x}_0)]=\bar f_0$, and $\hat f(\mathbf{x}_0)-\bar f_0=K^{-1}\sum_{i\in\mathcal{N}_0}\varepsilon_i$ is uncorrelated with $\varepsilon_0$. The squares have means $\sigma^2$, $\{f(\mathbf{x}_0)-\bar f_0\}^2$ and $\sigma^2/K$. ∎

**Proposition 6.33** (Average variance of the two fits; ISLR §3.5). Under the assumptions of Theorem 6.32, let $[\mathbf{1}\ \mathbf{X}]$ have full column rank $p+1$, and let $\hat y_i$ be the least squares fitted values. Then $n^{-1}\sum_i\operatorname{var}(\hat y_i)=\sigma^2(p+1)/n$, while the KNN fit at each training input has variance $\sigma^2/K$; the two agree at $K=n/(p+1)$.

**Proof.** $\operatorname{var}(\hat{\mathbf{Y}})=\sigma^2\mathbf{H}\mathbf{H}'=\sigma^2\mathbf{H}$ by the prerequisites' proposition **variance under a linear map**, $\mathbf{H}$ being symmetric and idempotent, so $\operatorname{var}(\hat y_i)=\sigma^2h_{ii}$; and $\sum_ih_{ii}=\operatorname{tr}(\mathbf{H})=p+1$ by Unit 2's theorem **properties of $\mathbf{H}$ and $\mathbf{M}$**. The KNN variance is that of Theorem 6.32, the same at every point. ∎

**Corollary 6.34** (The neighbourhood radius bounds the bias; ISLR §3.5). If $|f(\mathbf{u})-f(\mathbf{v})|\leqslant L\|\mathbf{u}-\mathbf{v}\|_\infty$ for all $\mathbf{u},\mathbf{v}$, then $|f(\mathbf{x}_0)-\bar f_0|\leqslant L\max_{i\in\mathcal{N}_0}\|\mathbf{x}_i-\mathbf{x}_0\|_\infty$, where $\bar f_0=K^{-1}\sum_{i\in\mathcal{N}_0}f(\mathbf{x}_i)$; the squared bias of Theorem 6.32 is at most the square of this bound.

**Proof.** $|f(\mathbf{x}_0)-\bar f_0|=|K^{-1}\sum_{i\in\mathcal{N}_0}\{f(\mathbf{x}_0)-f(\mathbf{x}_i)\}|$ is at most the average of $L\|\mathbf{x}_0-\mathbf{x}_i\|_\infty$, hence at most its maximum. ∎

**Remark** (ISLR §3.5). $K$ trades variance against bias: the variance $\sigma^2/K$ falls as $K$ grows, while the neighbour sets grow and the bound of Corollary 6.34 does not decrease. When $f$ is linear, least squares is unbiased (Unit 2's theorem **unbiasedness**); at a training input its expected error is $\sigma^2+\operatorname{var}(\hat y_i)$, by the expansion in the proof of Theorem 6.32, and the average $\sigma^2+\sigma^2(p+1)/n$ (Proposition 6.33) is at most the $\sigma^2+\sigma^2/K$ of KNN, before its squared bias, whenever $K\leqslant n/(p+1)$.

### 4.2 The curse of dimensionality

**Theorem 6.35** (Distance to the nearest neighbours; ISLR §3.5, §6.4.3). Let $\mathbf{x}_1,\ldots,\mathbf{x}_n$ be independent and uniform on the cube $C=[-\tfrac12,\tfrac12]^p$, and let $R_K$ be the $K$th smallest of $\|\mathbf{x}_1\|_\infty,\ldots,\|\mathbf{x}_n\|_\infty$, the distance from the centre of $C$ to its $K$th nearest neighbour in the sup norm. For $0\leqslant r\leqslant\tfrac12$, $\operatorname{\mathsf{P}}(R_K\leqslant r)=\operatorname{\mathsf{P}}\{\mathrm{Bin}(n,(2r)^p)\geqslant K\}$, and the median of $R_1$ is $m(n,p)=\tfrac12(1-2^{-1/n})^{1/p}$.

**Proof.** The coordinates of $\mathbf{x}_i$ are independent and uniform on $[-\tfrac12,\tfrac12]$, so $\operatorname{\mathsf{P}}(\|\mathbf{x}_i\|_\infty\leqslant r)=\prod_j\operatorname{\mathsf{P}}(|x_{ij}|\leqslant r)=(2r)^p$. These events are independent across $i$, so the number of points within distance $r$ is binomial with $n$ trials and success probability $(2r)^p$, and $R_K\leqslant r$ exactly when it is at least $K$. For $K=1$, $\operatorname{\mathsf{P}}(R_1>r)=\{1-(2r)^p\}^n$, which equals $\tfrac12$ at $(2r)^p=1-2^{-1/n}$; it is continuous and strictly decreasing in $r$, so this is the median. ∎

**Proposition 6.36** (Neighbourhoods are not local; ISLR §3.5). A cube of side $\ell\leqslant1$ inside $C$ has probability $\ell^p$ under the uniform law on $C$, so a cube holding a fraction $q$ of the points on average has side $q^{1/p}$.

**Proof.** By independence of the coordinates the probability is $\prod_{j\leqslant p}\ell=\ell^p$, and the expected fraction of $n$ points in it is the probability. ∎

**Example 6.8** (How far the nearest neighbour is; ISLR §3.5). The median distance $m(n,p)$ of Theorem 6.35, checked by simulation at $n=500$, $p=10$, and the side of a cube holding a fraction $q$ of the points:

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 3)
med <- function(n, p) 0.5 * (1 - 2^(-1 / n))^(1 / p)
tb <- outer(c(100, 500, 5000), c(1, 2, 5, 10, 20), Vectorize(med))
dimnames(tb) <- list(paste0("n=", c(100, 500, 5000)), paste0("p=", c(1, 2, 5, 10, 20))); round(tb, 4)
set.seed(2); sim_m <- replicate(4000, min(apply(abs(matrix(runif(5000, -0.5, 0.5), 500, 10)), 1, max)))
c(simulated_median = median(sim_m), formula = med(500, 10))
side <- outer(c(0.01, 0.1, 0.5), c(1, 2, 5, 10, 20), function(q, p) q^(1 / p))
dimnames(side) <- list(paste0("q=", c(0.01, 0.1, 0.5)), paste0("p=", c(1, 2, 5, 10, 20))); round(side, 3)

In one dimension the nearest of $500$ points lies at median sup-norm distance $0.0007$ from the centre; in ten dimensions at $0.2589$, more than half way to the boundary of a cube of half-width $0.5$, and $5000$ points bring it down only to $0.2057$. The simulated median, $0.2591$, agrees with the formula. In ten dimensions a cube holding a tenth of the points spans $0.794$ of the range of every coordinate.

### 4.3 KNN against least squares in simulation

**Example 6.9** (Linear and non-linear truths; ISLR §3.5, Figures 3.17–3.19). $n=50$ training points with $x\sim U(-1,1)$ and $\sigma=0.3$, for a linear truth $1+2x$, a slightly non-linear $1+2x+0.8x^2$ and a strongly non-linear $1+\sin(4x)+x$; the test error is averaged over $100$ training sets and $1000$ test points each:

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 2.4)
fs <- list(linear = function(x) 1 + 2 * x, slight = function(x) 1 + 2 * x + 0.8 * x^2,
           strong = function(x) 1 + sin(4 * x) + x)
Ks <- c(1, 2, 3, 5, 9, 15, 25)
run_knn <- function(f, p = 1, R = 100, ntr = 50, nte = 1000, sg = 0.3) {
  rowMeans(replicate(R, { Xtr <- matrix(runif(ntr * p, -1, 1), ntr); Xte <- matrix(runif(nte * p, -1, 1), nte)
    ytr <- f(Xtr[, 1]) + rnorm(ntr, sd = sg); yte <- f(Xte[, 1]) + rnorm(nte, sd = sg)
    b <- coef(lm(ytr ~ Xtr))
    c(LS = mean((yte - cbind(1, Xte) %*% b)^2),
      sapply(Ks, function(K) mean((yte - knn.reg(Xtr, Xte, ytr, k = K)$pred)^2))) })) }
set.seed(35); tab_1d <- t(sapply(fs, run_knn)); colnames(tab_1d) <- c("LS", paste0("K=", Ks)); round(tab_1d, 4)
set.seed(317); x1 <- runif(50, -1, 1); y1 <- fs$strong(x1) + rnorm(50, sd = 0.3); xg <- seq(-1, 1, length = 400)
op <- par(mfrow = c(1, 2), mar = c(3.2, 3.4, 0.6, 0.6), mgp = c(2, 0.6, 0))
plot(x1, y1, pch = 16, cex = 0.5, xlab = "x", ylab = "y"); curve(fs$strong(x), add = TRUE, lty = 2)
lines(xg, knn.reg(matrix(x1), matrix(xg), y1, k = 1)$pred, col = 4); lines(xg, knn.reg(matrix(x1), matrix(xg), y1, k = 9)$pred, col = 2)
abline(lm(y1 ~ x1), col = "grey40")
matplot(1 / Ks, t(tab_1d[, -1]), type = "b", pch = 16, cex = 0.5, lty = 1, col = c(1, 3, 6), log = "xy",
        ylim = range(tab_1d), xlab = "1/K", ylab = "test MSE")
abline(h = tab_1d[, 1], col = c(1, 3, 6), lty = 2)
par(op)

*Figure 6.8.* Left: one training set from the strongly non-linear truth (dashed), with the KNN fits for $K=1$ (blue) and $K=9$ (red) and the least squares line (grey). Right: test error of KNN against $1/K$ (solid) and of least squares (dashed), on log scales, for the linear (black), slightly non-linear (green) and strongly non-linear (purple) truths (ISLR Figures 3.17–3.19).

For the linear truth least squares has the smallest test error, $0.0929$, against $0.1152$ for the best $K$; for the slightly non-linear truth KNN with $K$ from $2$ to $9$ beats least squares ($0.1526$); for the strongly non-linear truth every $K$ does, $0.1222$ at $K=5$ against $0.5095$. In each row the KNN error first falls and then rises with $K$, as the variance $\sigma^2/K$ of Theorem 6.32 falls and the neighbourhoods that bound the bias in Corollary 6.34 widen.

**Example 6.10** (Noise predictors; ISLR §3.5, Figure 3.20). The strongly non-linear truth in $x_1$, with $p-1$ further predictors, uniform on $(-1,1)$, that the response does not depend on:

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 3)
set.seed(320)
tab_p <- t(sapply(c(1, 2, 3, 4, 10, 20), function(p) run_knn(fs$strong, p = p)))
dimnames(tab_p) <- list(paste0("p=", c(1, 2, 3, 4, 10, 20)), c("LS", paste0("K=", Ks))); round(tab_p, 4)

Up to $p=4$ some $K$ beats least squares ($0.4891$ against $0.5413$ at $p=4$); with $10$ and $20$ predictors least squares wins ($0.6235$ against at best $0.7751$, and $0.8468$ against $0.8988$). From $p=1$ to $p=20$ the least squares error rises from $0.5059$ to $0.8468$ and the best KNN error from $0.1195$ to $0.8988$. The noise predictors spread the neighbours out, as Theorem 6.35 quantifies for the sup norm, and with them the bound of Corollary 6.34, while the KNN variance $\sigma^2/K$ does not depend on $p$; the least squares variance $\sigma^2(p+1)/n$ of Proposition 6.33 grows with $p$.

**Remark** (ISLR §3.5). When $f$ is far from linear, least squares keeps a bias and KNN can win, as in Example 6.9; as predictors are added the neighbourhoods spread and least squares wins again, as in Example 6.10: parametric methods tend to do better when there are few observations per predictor. Even when KNN predicts slightly better, least squares may be preferred for its few interpretable coefficients and their tests.

## 5 High Dimensions

> **Reading.** ISLR §6.4. SRM learning outcome 2(g).

### 5.1 What goes wrong

**Definition 6.37** (High-dimensional data; ISLR §6.4.1). Data with more predictors than observations, $p>n$, are **high-dimensional**; ISLR extends the term to $p$ slightly smaller than $n$.

**Theorem 6.38** (A perfect fit carries no information; ISLR §6.4.2). Let $\tilde{\mathbf{X}}=[\mathbf{1}\ \mathbf{X}]$ and let $\mathbf{H}$ be the projection onto $\mathcal{C}(\tilde{\mathbf{X}})$. If $\operatorname{rank}(\tilde{\mathbf{X}})=n$, then $\mathbf{H}=\mathbf{I}_n$: every residual is zero, $\mathrm{SSE}=0$ and, when $\mathrm{SST}>0$, $R^2=1$, whatever the response.

**Proof.** $\mathcal{C}(\tilde{\mathbf{X}})$ is a subspace of $\mathbb{R}^n$ of dimension $n$, hence $\mathbb{R}^n$ itself, and the projection onto $\mathbb{R}^n$ is the identity (the prerequisites' theorem **orthogonal projection**). So $\hat{\mathbf{Y}}=\mathbf{Y}$, $\mathrm{SSE}=0$, and $R^2=1-\mathrm{SSE}/\mathrm{SST}=1$. ∎

**Corollary 6.39** (The criteria fail; ISLR §6.4.2). Under Theorem 6.38 with $p+1\geqslant n$, $\hat\sigma^2=\mathrm{SSE}/(n-p-1)$ is $0/0$ if $p+1=n$ and has a negative divisor if $p+1>n$, so $C_p$, AIC and BIC, which need $\hat\sigma^2>0$ from the model with all predictors, cannot be formed; $R_a^2$ of a model with $n-1$ predictors and rank $n$ is $0/0$. Along any nested sequence of models whose column spaces grow to $\mathbb{R}^n$, $R^2$ does not decrease and reaches $1$, whatever the response with $\mathrm{SST}>0$.

**Proof.** The first statements are Theorem 6.38 and Unit 5's definition **$C_p$, AIC, BIC and adjusted $R^2$**. Along the sequence, $\mathrm{SSE}$ is the minimum of $\|\mathbf{Y}-\mathbf{u}\|^2$ over the column space, by the prerequisites' theorem **orthogonal projection**, a minimum over a growing set; so $\mathrm{SSE}$ does not increase, $R^2=1-\mathrm{SSE}/\mathrm{SST}$ does not decrease, and its final value is Theorem 6.38. ∎

**Proposition 6.40** (Least squares is not unique; ISLR §6.4.2). The least squares solutions form $\hat{\boldsymbol{\beta}}+\mathcal{N}(\tilde{\mathbf{X}})$, an affine subspace of dimension $p+1-\operatorname{rank}(\tilde{\mathbf{X}})$, all with the same fitted values; if $p+1>n$ its dimension is at least $p+1-n>0$.

**Proof.** $\|\mathbf{Y}-\tilde{\mathbf{X}}\boldsymbol{\beta}\|^2$ depends on $\boldsymbol{\beta}$ through $\tilde{\mathbf{X}}\boldsymbol{\beta}$ only, and is minimal exactly when $\tilde{\mathbf{X}}\boldsymbol{\beta}=\mathbf{H}\mathbf{Y}$, the unique point of $\mathcal{C}(\tilde{\mathbf{X}})$ nearest $\mathbf{Y}$ by the prerequisites' theorem **orthogonal projection**. The solutions of $\tilde{\mathbf{X}}\boldsymbol{\beta}=\mathbf{H}\mathbf{Y}$ form $\hat{\boldsymbol{\beta}}+\mathcal{N}(\tilde{\mathbf{X}})$, of dimension $p+1-\operatorname{rank}(\tilde{\mathbf{X}})$ by the prerequisites' theorem **rank–nullity**, and $\operatorname{rank}(\tilde{\mathbf{X}})\leqslant n$. ∎

**Proposition 6.41** (Regularised fits in high dimensions; ISLR §6.4.3, §6.4.4). Let $\mathbf{X}$ be centred and $\lambda>0$, with $n$ and $p$ arbitrary.

(i) The ridge estimate exists and is unique (Theorem 6.3), with $\operatorname{df}(\lambda)<1+\operatorname{rank}(\mathbf{X})\leqslant n$ if $\operatorname{rank}(\mathbf{X})\geqslant1$ (Proposition 6.12).

(ii) Lasso estimators exist and share one fit (Theorem 6.15), and one of them has at most $\operatorname{rank}(\mathbf{X})\leqslant\min(n-1,p)$ nonzero coefficients (Theorem 6.20).

(iii) If $p>n-1$, the columns of $\mathbf{X}$ are linearly dependent: some predictor is an exact linear combination of the others.

**Proof.** (i) and (ii) restate the cited results, with $d_j^2/(d_j^2+\lambda)<1$ for every $j$ in (i). For (iii), the centred columns lie in the orthogonal complement of $\mathbf{1}$, of dimension $n-1$, so more than $n-1$ of them are dependent, and a column with a nonzero coefficient in a vanishing combination is a combination of the others. ∎

**Remark** (ISLR §6.4.4). By Proposition 6.41(iii), when $p\geqslant n$ some predictor is an exact combination of the others, so $\boldsymbol{\beta}$ and $\boldsymbol{\beta}+\mathbf{v}$ with $\mathbf{X}\mathbf{v}=\mathbf{0}\neq\mathbf{v}$ give the same mean $\mathbf{X}\boldsymbol{\beta}$ and no data can tell them apart: at best a model is found that predicts well, not the variables that drive the response. When $\operatorname{rank}(\tilde{\mathbf{X}})=n$, training $\mathrm{SSE}$, $p$-values and $R^2$ are no evidence of a good fit, by Theorem 6.38; errors are reported on an independent test set or by cross-validation.

### 5.2 High dimensions in simulation

**Example 6.11** (Unrelated features; ISLR §6.4.2, Figure 6.23). $n=20$ training observations, a response unrelated to $19$ standard normal features, and least squares on the first $k$ of them, with the test error on $1000$ new observations:

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 2.3)
set.seed(623); n_u <- 20; P_u <- 19
X_u <- matrix(rnorm(n_u * P_u), n_u); y_u <- rnorm(n_u)
Xt_u <- matrix(rnorm(1000 * P_u), 1000); yt_u <- rnorm(1000)
res_u <- t(sapply(1:P_u, function(k) { f <- lm(y_u ~ X_u[, 1:k, drop = FALSE]); b <- coef(f)
  c(R2 = summary(f)$r.squared, train_MSE = mean(residuals(f)^2),
    test_MSE = mean((yt_u - cbind(1, Xt_u[, 1:k, drop = FALSE]) %*% b)^2)) }))
round(res_u[c(1, 5, 10, 15, 18, 19), ], 4)
op <- par(mfrow = c(1, 3), mar = c(3.2, 3.6, 0.6, 0.4), mgp = c(2, 0.6, 0))
for (v in colnames(res_u)) plot(1:P_u, res_u[, v], type = "b", pch = 16, cex = 0.5,
                                xlab = "number of features", ylab = v)
par(op)

*Figure 6.9.* $R^2$, training MSE and test MSE of least squares on $n=20$ observations as features unrelated to the response are added (ISLR Figure 6.23).

$R^2$ rises to $1$ and the training error falls to $0$ at $19$ features, where $20$ coefficients fit $20$ observations exactly (Theorem 6.38), while the test error rises from $1.337$ to $17.405$.

**Example 6.12** (The lasso as $p$ grows; ISLR §6.4.3, Figure 6.24). $n=100$ training observations, $p=20$, $50$ or $2000$ standard normal features of which $20$ have coefficients $\pm1$, $\sigma=1$; the lasso at the points of `glmnet`'s grid nearest $0.5$, $0.2$ and $0.05$ times $\lambda_{\max}$, averaged over $20$ training sets, reporting the test error on $500$ new observations and the number of nonzero coefficients, which ISLR calls the lasso's degrees of freedom:

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 3)
sim_p <- function(p, R = 20) {
  out <- replicate(R, { X <- matrix(rnorm(100 * p), 100); b <- c(rep(1, 20), rep(0, p - 20)) * sample(c(-1, 1), p, TRUE)
    y <- drop(X %*% b + rnorm(100)); Xt <- matrix(rnorm(500 * p), 500); yt <- drop(Xt %*% b + rnorm(500))
    f <- glmnet(X, y, alpha = 1); lam <- f$lambda
    ks <- sapply(c(0.5, 0.2, 0.05), function(q) which.min(abs(lam / max(lam) - q)))
    rbind(test_MSE = colMeans((yt - predict(f, Xt, s = lam[ks]))^2), df = f$df[ks]) })
  m <- apply(out, c(1, 2), mean); colnames(m) <- c("0.5", "0.2", "0.05"); m }
set.seed(624); lapply(c(p20 = 20, p50 = 50, p2000 = 2000), function(p) round(sim_p(p), 2))

With $p=20$ and $p=50$ the test error falls from $16.92$ and $16.97$ to $1.67$ and $1.82$ as $\lambda$ decreases to $0.05\lambda_{\max}$; with $p=2000$ it lies between $16.29$ and $18.04$ at all three $\lambda$, the fitted models using $21.90$ to $90.30$ predictors on average. Noise features make every fit worse, ISLR's curse of dimensionality; unlike ISLR's example, the least regularised of the three fits is best at $p=50$ here.

**Example 6.13** (One of many models; ISLR §6.4.4). The lasso on `Hitters`, refitted to $100$ bootstrap samples with $\lambda_{1\mathrm{SE}}$ chosen by ten-fold cross-validation each time; the proportion of samples in which each predictor is selected:

In [ ]:
H6 <- na.omit(Hitters); xH6 <- model.matrix(Salary ~ ., H6)[, -1]; yH6 <- H6$Salary
set.seed(8)
sel <- replicate(100, { ii <- sample(nrow(xH6), replace = TRUE)
  as.numeric(as.matrix(coef(cv.glmnet(xH6[ii, ], yH6[ii], alpha = 1), s = "lambda.1se"))[-1, 1] != 0) })
fr <- setNames(rowMeans(sel), colnames(xH6)); round(sort(fr, decreasing = TRUE), 2)
round(c(mean_size = mean(colSums(sel)), sd_size = sd(colSums(sel))), 2)
c(min_size = min(colSums(sel)), max_size = max(colSums(sel)), always_selected = sum(fr == 1))

No predictor is selected in all $100$ samples; `Hits` ($0.89$), `Walks` ($0.85$) and `PutOuts` ($0.80$) are selected most often, and the selected sets have $6.62$ predictors on average, from $2$ to $14$: the selected set changes from sample to sample.

## 6 Computation in R

> **Reading.** ISLR §6.5.2.

### 6.1 The ISLR lab: ridge regression and the lasso

In [ ]:
Hitters <- na.omit(Hitters)
x <- model.matrix(Salary ~ ., Hitters)[, -1]
y <- Hitters$Salary
grid <- 10^seq(10, -2, length = 100)
ridge.mod <- glmnet(x, y, alpha = 0, lambda = grid)
dim(coef(ridge.mod))
ridge.mod$lambda[50]; sqrt(sum(coef(ridge.mod)[-1, 50]^2))
ridge.mod$lambda[60]; sqrt(sum(coef(ridge.mod)[-1, 60]^2))
round(predict(ridge.mod, s = 50, type = "coefficients")[1:20, ], 3)

`glmnet` standardises the predictors by default and reports coefficients on the original scale; $\lambda'$ is its `lambda`, as in Example 6.4. At $\lambda'=11{,}498$ the $\ell_2$ norm of the coefficients is $6.361$, and at $\lambda'=705.5$ it is $57.11$: the larger $\lambda'$ shrinks more, although Proposition 6.7 guarantees a monotone norm only for the standardised coefficients, not for these. A training half, and the test error at $\lambda'=4$, at the null model, at $\lambda'=10^{10}$ and at least squares, by `glmnet` and by `lm`:

In [ ]:
set.seed(1)
train <- sample(1:nrow(x), nrow(x) / 2)
test <- (-train)
y.test <- y[test]
ridge.mod <- glmnet(x[train, ], y[train], alpha = 0, lambda = grid, thresh = 1e-12)
ridge.pred <- predict(ridge.mod, s = 4, newx = x[test, ])
mean((ridge.pred - y.test)^2)
mean((mean(y[train]) - y.test)^2)
ridge.pred <- predict(ridge.mod, s = 1e10, newx = x[test, ])
mean((ridge.pred - y.test)^2)
ridge.pred <- predict(ridge.mod, s = 0, newx = x[test, ], exact = T, x = x[train, ], y = y[train])
mean((ridge.pred - y.test)^2)
mean((cbind(1, x[test, ]) %*% coef(lm(y ~ x, subset = train)) - y.test)^2)
round(rbind(lm = coef(lm(y ~ x, subset = train)),
  glmnet_exact = predict(ridge.mod, s = 0, exact = T, type = "coefficients", x = x[train, ], y = y[train])[1:20, ]), 3)[, 1:8]

The test error is $142{,}199$ at $\lambda'=4$, against $224{,}670$ for the null model, which $\lambda'=10^{10}$ reproduces, and $168{,}589$ for the least squares fit that `glmnet` computes at $\lambda'=0$ with `exact = T`, against $168{,}593$ from `lm`; the coefficients agree to about three significant digits (`Hits`: $-1.637$, against $-1.638$ from `lm`).

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 2.4)
set.seed(1)
cv.out <- cv.glmnet(x[train, ], y[train], alpha = 0)
op <- par(mar = c(3.2, 3.8, 1.8, 0.6), mgp = c(2.1, 0.6, 0)); plot(cv.out); par(op)
bestlam <- cv.out$lambda.min
bestlam
ridge.pred <- predict(ridge.mod, s = bestlam, newx = x[test, ])
mean((ridge.pred - y.test)^2)
out <- glmnet(x, y, alpha = 0)
round(predict(out, type = "coefficients", s = bestlam)[1:20, ], 3)

*Figure 6.10.* Ten-fold cross-validation error of ridge regression on the `Hitters` training half, with $\lambda_{\min}$ and $\lambda_{1\mathrm{SE}}$ dotted.

Cross-validation selects $\lambda'=326.1$, with test error $139{,}857$, lower than at $\lambda'=4$; refitted on all the data, ridge keeps all $19$ coefficients nonzero.

In [ ]:
lasso.mod <- glmnet(x[train, ], y[train], alpha = 1, lambda = grid)
set.seed(1)
cv.out <- cv.glmnet(x[train, ], y[train], alpha = 1)
op <- par(mfrow = c(1, 2), mar = c(3.2, 3.8, 1.8, 0.6), mgp = c(2.1, 0.6, 0))
plot(lasso.mod, xvar = "norm"); plot(cv.out); par(op)
bestlam <- cv.out$lambda.min
lasso.pred <- predict(lasso.mod, s = bestlam, newx = x[test, ])
mean((lasso.pred - y.test)^2)
out <- glmnet(x, y, alpha = 1, lambda = grid)
lasso.coef <- predict(out, type = "coefficients", s = bestlam)[1:20, ]
round(lasso.coef[lasso.coef != 0], 3)

*Figure 6.11.* Lasso coefficient paths on the `Hitters` training half against their $\ell_1$ norm, and the ten-fold cross-validation error.

At its cross-validated $\lambda'$ the lasso has test error $143{,}674$, close to that of ridge, and its fit to all the data keeps $11$ of the $19$ predictors, as in the lab.

### 6.2 Formulas against package output

On the standardised `Hitters` predictors (divisor $n$) and the centred response: ridge by Theorem 6.3, by Theorem 6.5 and by `MASS::lm.ridge`; the $\lambda$ that `glmnet`'s ridge actually applies, recovered from the equation of Theorem 6.3 coordinate by coordinate; the degrees of freedom of Proposition 6.12; and the risk formula of Theorem 6.10 against simulation:

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 3)
nH <- nrow(x); pH <- ncol(x); Xs <- scale(x, center = TRUE, scale = apply(x, 2, sdn)); ycH <- y - mean(y)
lam <- 100; svH <- svd(Xs); dH <- svH$d
b_solve <- drop(solve(crossprod(Xs) + lam * diag(pH), crossprod(Xs, ycH)))
b_svd <- drop(svH$v %*% (dH / (dH^2 + lam) * crossprod(svH$u, ycH)))
b_mass <- coef(MASS::lm.ridge(Salary ~ ., data = Hitters, lambda = lam))[-1] * apply(x, 2, sdn)
lg <- 0.38
b_glm <- as.matrix(coef(glmnet(Xs, ycH, alpha = 0, lambda = lg * c(4, 2, 1), standardize = FALSE,
                               intercept = FALSE, thresh = 1e-16)))[-1, 3]
implied <- drop(crossprod(Xs, ycH - Xs %*% b_glm)) / b_glm
S_H <- Xs %*% solve(crossprod(Xs) + lam * diag(pH)) %*% t(Xs) + matrix(1 / nH, nH, nH)
signif(c(solve_vs_svd = max(abs(b_solve - b_svd)), solve_vs_lm.ridge = max(abs(b_solve - b_mass)),
  df_trace_vs_formula = sum(diag(S_H)) - (1 + sum(dH^2 / (dH^2 + lam)))), 3)
round(c(implied_lambda_mean = mean(implied), implied_lambda_sd = sd(implied), n_times_glmnet = nH * lg,
  n_times_glmnet_over_sd_y = nH * lg / sdn(y), df = sum(diag(S_H))), 3)
bet <- b_solve; s2 <- summary(lm(Salary ~ ., data = Hitters))$sigma^2
gam <- dH^2; al <- drop(crossprod(svH$v, bet))
R_fun <- function(l) sum(s2 * gam / (gam + l)^2) + sum(l^2 * al^2 / (gam + l)^2)
set.seed(4); W50 <- solve(crossprod(Xs) + 50 * diag(pH), t(Xs)); mu <- drop(Xs %*% bet)
sim_R <- replicate(2000, sum((drop(W50 %*% (mu + rnorm(nH, sd = sqrt(s2)))) - bet)^2))
round(c(formula_R50 = R_fun(50), simulated_R50 = mean(sim_R), mc_se = sd(sim_R) / sqrt(2000),
  R0 = R_fun(0), argmin = optimize(R_fun, c(0, 3000))$minimum), 1)

The three ridge computations agree to rounding error. The ridge fit of `glmnet` at $\lambda'=0.38$ satisfies the equation of Theorem 6.3 with the same $\lambda=0.222$ at every coordinate, which is $n\lambda'/\mathrm{sd}_n(y)$ and not $n\lambda'=99.940$. At $\lambda=100$ the degrees of freedom are $8.960$. The risk formula gives $R(50)=7570.8$ against $7517.6$ simulated, with standard error $57.7$; least squares has $R(0)=466{,}647.2$, and $R$ is smallest near $\lambda=160.5$. The lasso by coordinate descent (Proposition 6.25) against `glmnet` with $\lambda'=\lambda/(2n)$, the conditions of Theorem 6.18, $\lambda_{\max}$ of Corollary 6.19 against the first $\lambda$ of `glmnet`'s own path, and the cross-validation curve of Definition 6.29 rebuilt from fold fits:

In [ ]:
lasso_cd <- function(X, y, lam, tol = 1e-12) { b <- numeric(ncol(X)); ss <- colSums(X^2); r <- y
  repeat { old <- b
    for (j in seq_along(b)) { r <- r + X[, j] * b[j]; b[j] <- soft(sum(X[, j] * r), lam / 2) / ss[j]; r <- r - X[, j] * b[j] }
    if (max(abs(b - old)) < tol) break }
  b }
lam1 <- 1000; bcd <- lasso_cd(Xs, ycH, lam1)
bgl <- as.matrix(coef(glmnet(Xs, ycH, alpha = 1, lambda = lam1 / (2 * nH) * c(4, 2, 1), standardize = FALSE,
                             intercept = FALSE, thresh = 1e-16)))[-1, 3]
cj <- drop(crossprod(Xs, ycH - Xs %*% bcd)); on <- bcd != 0
path0 <- glmnet(Xs, ycH, alpha = 1, standardize = FALSE, intercept = FALSE)
signif(c(cd_vs_glmnet = max(abs(bcd - bgl)), largest_coefficient = max(abs(bcd)), active = sum(on),
  active_condition = max(abs(2 * cj[on] - lam1 * sign(bcd[on]))), inactive_max_abs_2c = max(abs(2 * cj[!on])),
  lambda_max = 2 * max(abs(crossprod(Xs, ycH))), two_n_first_glmnet_lambda = 2 * nH * path0$lambda[1]), 4)
set.seed(1); fold <- sample(rep(1:10, length.out = nH))
cvg <- cv.glmnet(x, y, alpha = 1, foldid = fold); lgrid <- cvg$lambda
P <- matrix(NA_real_, nH, length(lgrid))
for (v in 1:10) P[fold == v, ] <- predict(glmnet(x[fold != v, ], y[fold != v], alpha = 1), x[fold == v, , drop = FALSE], s = lgrid)
cvm <- colMeans((y - P)^2)
fm <- t(sapply(1:10, function(v) colMeans((y[fold == v] - P[fold == v, , drop = FALSE])^2)))
cvsd <- sqrt(apply(sweep(fm, 2, cvm)^2, 2, weighted.mean, w = as.vector(table(fold))) / 9)
i_min <- which.min(cvm)
signif(c(cv_curve = max(abs(cvm - cvg$cvm)), cv_se = max(abs(cvsd - cvg$cvsd)),
  lambda_min = lgrid[i_min] - cvg$lambda.min, lambda_1se = max(lgrid[cvm <= cvm[i_min] + cvsd[i_min]]) - cvg$lambda.1se), 3)

Coordinate descent and `glmnet` agree to $2.788\times10^{-4}$ on coefficients as large as $276.7$. The $15$ active coefficients satisfy $2c_j=\lambda\operatorname{sign}(\hat\beta_j)$ to $4.212\times10^{-10}$, and the inactive ones have $|2c_j|\leqslant923.2<\lambda=1000$, as Theorem 6.18 requires; $2n$ times the first $\lambda'$ of `glmnet`'s path is $\lambda_{\max}=1.343\times10^5$. The rebuilt cross-validation curve, its standard errors and both selected $\lambda$ agree with `cv.glmnet`, each fold being fitted on its own grid and interpolated. KNN regression from Definition 6.31 against `FNN::knn.reg`:

In [ ]:
knn_manual <- function(xtr, ytr, xte, K) apply(xte, 1, function(x0)
  mean(ytr[order(sqrt(colSums((t(xtr) - x0)^2)))[1:K]]))
set.seed(7); xtr <- matrix(runif(400), 200, 2); ytr <- sin(3 * xtr[, 1]) + rnorm(200, sd = 0.2)
xte <- matrix(runif(100), 50, 2)
signif(sapply(c(K1 = 1, K5 = 5, K20 = 20), function(K)
  max(abs(knn_manual(xtr, ytr, xte, K) - knn.reg(xtr, xte, ytr, k = K)$pred))), 3)

The two agree exactly.

## 7 Exercises

**Exercise 6.1** (ISLR §3.7, exercise 2). Carefully explain the differences between the KNN classifier and KNN regression methods.

**Solution.** Both find the set $\mathcal{N}_0$ of the $K$ training points nearest the prediction point $\mathbf{x}_0$. The KNN classifier (Unit 1's definition **$K$-nearest neighbours**) applies to a qualitative response: it estimates each class probability by the proportion $K^{-1}\sum_{i\in\mathcal{N}_0}I(y_i=j)$ and predicts the class with the largest proportion, and it is judged by the error rate. KNN regression (Definition 6.31) applies to a quantitative response: it predicts the average $K^{-1}\sum_{i\in\mathcal{N}_0}y_i$, and it is judged by the mean squared error, which Theorem 6.32 splits into $\sigma^2$, a squared bias governed by the spread of $f$ over $\mathcal{N}_0$, and the variance $\sigma^2/K$. In both, $K$ sets the flexibility: $K=1$ reproduces the training responses, and large $K$ gives a smooth, biased fit. ■

**Exercise 6.2** (ISLR §6.6, exercise 2). For (a) the lasso, (b) ridge regression and (c) non-linear methods, each relative to least squares, which is correct? i. More flexible, and better prediction when its increase in bias is less than its decrease in variance. ii. More flexible, and better prediction when its increase in variance is less than its decrease in bias. iii. Less flexible, and better prediction when its increase in bias is less than its decrease in variance. iv. Less flexible, and better prediction when its increase in variance is less than its decrease in bias.

**Solution.** The expected test error is the irreducible error plus squared bias plus variance (Unit 1's proposition **bias–variance decomposition**), so a method predicts better than least squares exactly when its change in squared bias is smaller than the opposite change in variance. (a) iii: the lasso restricts the least squares fit, the diamond $\|\boldsymbol{\beta}\|_1\leqslant s$ of Theorem 6.16 replacing all of $\mathbb{R}^p$; its coefficients move towards zero, adding bias and removing variance (for an orthonormal design, Proposition 6.23). (b) iii: likewise for ridge, whose variance falls and squared bias rises with $\lambda$ (Theorem 6.10) and whose degrees of freedom fall below those of least squares (Proposition 6.12). (c) ii: a non-linear method contains the linear fits as special cases or approximates them, so it is more flexible, with less bias and more variance. ■

**Exercise 6.3** (ISLR §6.6, exercise 3). The lasso in its constrained form, $\mathrm{RSS}$ minimised subject to $\sum_j|\beta_j|\leqslant s$. As $s$ increases from $0$, does each of (a) the training $\mathrm{RSS}$, (b) the test $\mathrm{RSS}$, (c) the variance, (d) the squared bias, (e) the irreducible error: i. increase initially and then decrease in an inverted U; ii. decrease initially and then increase in a U; iii. steadily increase; iv. steadily decrease; v. remain constant?

**Solution.** (a) iv: the feasible set grows with $s$, so the minimum of $\mathrm{RSS}$ cannot increase, and it reaches the least squares value once $s\geqslant\|\hat{\boldsymbol{\beta}}\|_1$. (e) v: $\operatorname{var}(\varepsilon)$ does not depend on the fit. (c) iii and (d) iv, a larger budget allowing a more flexible fit; for one centred predictor $\mathbf{x}$ with $\mathbf{x}'\mathbf{x}=1$ this is proved as follows. By Pythagoras, as in the proof of Theorem 6.22, $\|\mathbf{Y}_c-\mathbf{x} b\|^2=\|\mathbf{Y}_c-\mathbf{x} z\|^2+(b-z)^2$ with $z=\mathbf{x}'\mathbf{Y}_c$, so the constrained lasso is the point of $[-s,s]$ nearest $z$, $c_s(z)=\operatorname{sign}(z)\min(|z|,s)$; under A1–A5, $z\sim N(\theta,\sigma^2)$ with $\theta=\beta_1$ (Step 4 of the proof of Proposition 6.23). With the $g$ and $k$ of Step 1 of that proof and $a_\mp=(s\mp\theta)/\sigma$, the identities $c_s(z)=z-(z-s)^++(-z-s)^+$ and $z^2-c_s(z)^2=\{(z-s)^+\}^2+2s(z-s)^++\{(-z-s)^+\}^2+2s(-z-s)^+$ give

$$
\begin{align*}
\operatorname{\mathsf{E}} c_s(z)&=\theta-\sigma g(a_-)+\sigma g(a_+),\\
\operatorname{\mathsf{E}} c_s(z)^2&=\sigma^2+\theta^2-\sigma^2\{k(a_-)+k(a_+)\}-2s\sigma\{g(a_-)+g(a_+)\},
\end{align*}
$$

so, by $g'=-(1-\Phi)$ and $k'=-2g$, $\operatorname{\mathsf{E}} c_s(z)$ has derivative $\Phi(a_+)-\Phi(a_-)=\operatorname{\mathsf{P}}(z>s)-\operatorname{\mathsf{P}}(z<-s)$ in $s$ and $\operatorname{\mathsf{E}} c_s(z)^2$ has derivative $2s\operatorname{\mathsf{P}}(|z|>s)$. The derivative of the variance,

$$
2s\operatorname{\mathsf{P}}(|z|>s)-2\operatorname{\mathsf{E}}[c_s(z)]\{\operatorname{\mathsf{P}}(z>s)-\operatorname{\mathsf{P}}(z<-s)\} ,
$$

is nonnegative as $|\operatorname{\mathsf{E}} c_s(z)|\leqslant s$, so by the prerequisites' mean value theorem the variance does not decrease, (c). For $\theta\geqslant0$, $\operatorname{\mathsf{E}} c_s(z)-\theta=\sigma\{g(a_+)-g(a_-)\}\leqslant0$, $g$ being decreasing and $a_+\geqslant a_-$, while $\operatorname{\mathsf{E}} c_s(z)$, with derivative $\Phi(a_+)-\Phi(a_-)\geqslant0$, does not decrease; so the squared bias $\{\theta-\operatorname{\mathsf{E}} c_s(z)\}^2$ does not increase, (d), and $\theta<0$ follows from $c_s(-z)=-c_s(z)$. (b) ii: by the argument of the remark after Theorem 6.10, which holds for any estimator, the expected test error is $\sigma^2$ plus the error of the intercept plus the squared bias and the variance of the slopes; it starts at the null model and ends at least squares, and it typically falls first and rises later, as in Example 6.5, where it is minimal at an intermediate $\lambda$ for both the dense and the sparse model. ■

**Exercise 6.4** (ISLR §6.6, exercise 4). Ridge regression in its penalised form. As $\lambda$ increases from $0$, answer (a)–(e) of Exercise 6.3 with the same options.

**Solution.** (a) iii: by Theorem 6.5, with $\mathbf{P}=\mathbf{U}\mathbf{U}'$, $\mathbf{Y}_c-\mathbf{X}\hat{\boldsymbol{\beta}}_\lambda=\sum_j\{\lambda/(d_j^2+\lambda)\}(\mathbf{u}_j'\mathbf{Y})\mathbf{u}_j+(\mathbf{I}-\mathbf{P})\mathbf{Y}_c$, two orthogonal parts, so $\mathrm{RSS}(\lambda)=\sum_j\{\lambda/(d_j^2+\lambda)\}^2(\mathbf{u}_j'\mathbf{Y})^2+\|(\mathbf{I}-\mathbf{P})\mathbf{Y}_c\|^2$, which does not decrease in $\lambda$. (c) iv and (d) iii: Theorem 6.10, the variance sum decreasing and the squared bias sum increasing. (b) ii: the test error for isotropic test points is $\sigma^2+\operatorname{\mathsf{E}}(\hat\beta_0-\beta_0)^2+R(\lambda)$ (the remark after Theorem 6.10), and $R$ decreases near $\lambda=0$ and increases for large $\lambda$. (e) v, as before. ■

**Exercise 6.5** (ISLR §6.6, exercise 5). Let $n=p=2$, $x_{11}=x_{12}$, $x_{21}=x_{22}$, $y_1+y_2=0$, $x_{11}+x_{21}=0$ and $x_{12}+x_{22}=0$, so that the intercept is zero for least squares, ridge and the lasso. (a) Write the ridge problem. (b) Show that the ridge estimates satisfy $\hat\beta_1=\hat\beta_2$. (c) Write the lasso problem. (d) Show that the lasso estimates are not unique, and describe them.

**Solution.** Write $x_{11}=x_{12}=a\neq0$, $x_{21}=x_{22}=-a$, $y_1=y$, $y_2=-y$ and $t=\beta_1+\beta_2$; then $\mathrm{RSS}=(y-at)^2+(-y+at)^2=2(y-at)^2$. (a) Minimise $2(y-a\beta_1-a\beta_2)^2+\lambda(\beta_1^2+\beta_2^2)$. (b) For fixed $t$, $\beta_1^2+\beta_2^2=t^2/2+(\beta_1-\beta_2)^2/2$ is smallest at $\beta_1=\beta_2$; so the minimiser has $\hat\beta_1=\hat\beta_2=t/2$, with $t$ minimising $2(y-at)^2+\lambda t^2/2$, $t=4ay/(4a^2+\lambda)$: $\hat\beta_1=\hat\beta_2=2ay/(4a^2+\lambda)$, as Theorem 6.28 requires. (c) Minimise $2(y-a\beta_1-a\beta_2)^2+\lambda(|\beta_1|+|\beta_2|)$. (d) $|\beta_1|+|\beta_2|\geqslant|t|$, with equality iff $\beta_1\beta_2\geqslant0$; so the minimum is that of $2(y-at)^2+\lambda|t|=2a^2\{(t-y/a)^2+(\lambda/(2a^2))|t|\}$, attained by Lemma 6.21 at $t^\ast=S_{\lambda/(4a^2)}(y/a)$, and the lasso estimates are all $(\beta_1,\beta_2)$ with $\beta_1+\beta_2=t^\ast$ and $\beta_1\beta_2\geqslant0$: a segment from $(t^\ast,0)$ to $(0,t^\ast)$, a single point only when $t^\ast=0$. On the data $a=1$, $y=2$, $\lambda=1$:

In [ ]:
a5 <- 1; y5 <- 2; l5 <- 1
t_star <- sign(y5 / a5) * max(abs(y5 / a5) - l5 / (4 * a5^2), 0)
Lcrit <- function(b) 2 * (y5 - a5 * b[1] - a5 * b[2])^2 + l5 * sum(abs(b))
c(t_star = t_star, ridge_each = 2 * a5 * y5 / (4 * a5^2 + l5),
  lasso_at_t_0 = Lcrit(c(t_star, 0)), lasso_at_half_half = Lcrit(c(t_star, t_star) / 2),
  lasso_at_0_t = Lcrit(c(0, t_star)), off_segment = Lcrit(c(t_star + 0.3, -0.3)))

The three points of the segment give the same value, and a point with the same sum off the segment gives a larger one. ■

**Exercise 6.6** (ISLR §6.6, exercise 6). With $p=1$, plot the ridge criterion $(y_1-\beta_1)^2+\lambda\beta_1^2$ and the lasso criterion $(y_1-\beta_1)^2+\lambda|\beta_1|$ against $\beta_1$ for some $y_1$ and $\lambda>0$, and confirm that they are minimised at $y_1/(1+\lambda)$ and at the soft-thresholded value of Theorem 6.22.

**Solution.** With $y_1=2$ and $\lambda=1$ the minimisers are $1$ and $2-1/2=1.5$:

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 2.3)
y6 <- 2; l6 <- 1; bb <- seq(-1, 3, length = 401)
rc <- function(b) (y6 - b)^2 + l6 * b^2; lc <- function(b) (y6 - b)^2 + l6 * abs(b)
c(ridge_numeric = optimize(rc, c(-5, 5))$minimum, ridge_formula = y6 / (1 + l6),
  lasso_numeric = optimize(lc, c(-5, 5))$minimum, lasso_formula = sign(y6) * max(abs(y6) - l6 / 2, 0))
op <- par(mfrow = c(1, 2), mar = c(3.2, 3.4, 0.6, 0.6), mgp = c(2, 0.6, 0))
plot(bb, rc(bb), type = "l", xlab = "beta", ylab = "ridge criterion"); abline(v = y6 / (1 + l6), lty = 2)
plot(bb, lc(bb), type = "l", xlab = "beta", ylab = "lasso criterion"); abline(v = y6 - l6 / 2, lty = 2)
par(op)

*Figure 6.12.* The ridge and lasso criteria for $p=1$, $y_1=2$, $\lambda=1$, with the minimisers of Theorem 6.22 dashed.

The numerical minimisers agree with the formulas to the tolerance of `optimize`. ■

**Exercise 6.7** (ISLR §6.6, exercise 7). Let $y_i=\beta_0+\sum_jx_{ij}\beta_j+\varepsilon_i$ with $\varepsilon_i$ independent $N(0,\sigma^2)$. (a) Write the likelihood. (b) With $\beta_1,\ldots,\beta_p$ independent double-exponential with mean $0$ and scale $b$, $p(\beta)=e^{-|\beta|/b}/(2b)$, write the posterior. (c) Show that the lasso estimate is its mode. (d) With a normal prior of mean $0$ and variance $c$, write the posterior. (e) Show that the ridge estimate is both its mode and its mean.

**Solution.** (a) $f(\mathbf{Y}\mid\beta_0,\boldsymbol{\beta})=(2\pi\sigma^2)^{-n/2}\exp\{-\|\mathbf{Y}-\beta_0\mathbf{1}-\mathbf{X}\boldsymbol{\beta}\|^2/(2\sigma^2)\}$. The priors concern $\boldsymbol{\beta}$ only, so take $\beta_0$ as known; then $\mathbf{Y}-\beta_0\mathbf{1}$ follows the model of Proposition 6.24, and $\mathbf{Y}$ stands for $\mathbf{Y}-\beta_0\mathbf{1}$ below. (b) By Proposition 6.24, $\pi(\boldsymbol{\beta}\mid\mathbf{Y})\propto\exp\{-\|\mathbf{Y}-\mathbf{X}\boldsymbol{\beta}\|^2/(2\sigma^2)-\|\boldsymbol{\beta}\|_1/b\}$, the constant being $(2\pi\sigma^2)^{-n/2}(2b)^{-p}/q(\mathbf{Y})$. (c) The mode maximises the exponent, that is minimises $\|\mathbf{Y}-\mathbf{X}\boldsymbol{\beta}\|^2+(2\sigma^2/b)\|\boldsymbol{\beta}\|_1$: the lasso without intercept with $\lambda=2\sigma^2/b$ (Proposition 6.24(i) with $\omega=b$). (d) $\pi(\boldsymbol{\beta}\mid\mathbf{Y})\propto\exp\{-\|\mathbf{Y}-\mathbf{X}\boldsymbol{\beta}\|^2/(2\sigma^2)-\|\boldsymbol{\beta}\|^2/(2c)\}$, which is proportional to $\exp\{-(\boldsymbol{\beta}-\mathbf{m})'\mathbf{A}(\boldsymbol{\beta}-\mathbf{m})/(2\sigma^2)\}$ with $\mathbf{A}=\mathbf{X}'\mathbf{X}+(\sigma^2/c)\mathbf{I}$ and $\mathbf{m}=\mathbf{A}^{-1}\mathbf{X}'\mathbf{Y}$. (e) $\mathbf{m}$ is the ridge estimate without intercept with $\lambda=\sigma^2/c$, and Proposition 6.24(ii), with $\nu=c$, shows it is the mode and the mean. ■

**Exercise 6.8** (ISLR §6.6, exercise 8, parts (e) and (f)). With the data of Unit 5's exercise ISLR 6.6.8 ($X$ and $\varepsilon$ from `rnorm` with seed $1$, $Y=3+2X-3X^2+0.3X^3+\varepsilon$): (e) fit the lasso to $X,X^2,\ldots,X^{10}$ with $\lambda$ chosen by cross-validation, plot the cross-validation error, report the coefficients and discuss them; (f) generate $Y=3+7X^7+\varepsilon$ and fit the lasso.

**Solution.**

In [ ]:
set.seed(1); X8 <- rnorm(100); eps8 <- rnorm(100); Y8 <- 3 + 2 * X8 - 3 * X8^2 + 0.3 * X8^3 + eps8
M8 <- poly(X8, 10, raw = TRUE); colnames(M8) <- paste0("x", 1:10)
set.seed(1); cv8 <- cv.glmnet(M8, Y8, alpha = 1)
Y7 <- 3 + 7 * X8^7 + eps8; set.seed(1); cv7 <- cv.glmnet(M8, Y7, alpha = 1)
round(cbind(cubic_min = as.matrix(coef(cv8, s = "lambda.min"))[, 1], cubic_1se = as.matrix(coef(cv8, s = "lambda.1se"))[, 1],
            x7_min = as.matrix(coef(cv7, s = "lambda.min"))[, 1], x7_1se = as.matrix(coef(cv7, s = "lambda.1se"))[, 1]), 4)
op <- par(mfrow = c(1, 2), mar = c(3.2, 3.8, 1.8, 0.6), mgp = c(2.1, 0.6, 0)); plot(cv8); plot(cv7); par(op)

*Figure 6.13.* Ten-fold cross-validation error of the lasso for the cubic of part (e) and for the seventh-power model of part (f).

(e) At $\lambda_{\min}$ the lasso keeps $X$ and $X^2$ with coefficients $2.2242$ and $-3.0981$, close to $\beta_1=2$ and $\beta_2=-3$, and tiny ones on $X^5$ and $X^7$, but not the small cubic term; the one-standard-error fit adds $X^3$ with $0.1513$ and shrinks the rest. (f) For $Y=3+7X^7+\varepsilon$ both choices keep $X^7$ alone, with coefficient $6.797$ or $6.755$, shrunk from $7$, and intercept $3.820$ or $3.996$: the lasso finds the single relevant power, as BIC did in Unit 5's solution of this exercise. ■

**Exercise 6.9** (ISLR §6.6, exercise 9, parts (c), (d), (g)). Predict `Apps` in `College` with the training and test halves of Unit 5's exercise ISLR 6.6.9. (c) Fit ridge regression with $\lambda$ chosen by cross-validation and report the test error. (d) The same for the lasso, reporting also the number of nonzero coefficients. (g) Compare the test errors with those of least squares, PCR and PLS.

**Solution.**

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 3)
set.seed(1); tr9 <- sample(nrow(College), nrow(College) / 2); te9 <- College[-tr9, ]
xC <- model.matrix(Apps ~ ., College)[, -1]; yC <- College$Apps
set.seed(1); cvr9 <- cv.glmnet(xC[tr9, ], yC[tr9], alpha = 0)
set.seed(1); cvl9 <- cv.glmnet(xC[tr9, ], yC[tr9], alpha = 1)
set.seed(1); pcr9 <- pls::pcr(Apps ~ ., data = College, subset = tr9, scale = TRUE, validation = "CV")
set.seed(1); pls9 <- pls::plsr(Apps ~ ., data = College, subset = tr9, scale = TRUE, validation = "CV")
M9 <- c(PCR = unname(which.min(pls::MSEP(pcr9, estimate = "CV")$val[1, 1, -1])),
        PLS = unname(which.min(pls::MSEP(pls9, estimate = "CV")$val[1, 1, -1])))
pred9 <- list(least_squares = predict(lm(Apps ~ ., data = College, subset = tr9), te9),
              ridge = predict(cvr9, xC[-tr9, ], s = "lambda.min"), lasso = predict(cvl9, xC[-tr9, ], s = "lambda.min"),
              PCR = predict(pcr9, te9, ncomp = M9[["PCR"]]), PLS = predict(pls9, te9, ncomp = M9[["PLS"]]))
test9 <- sapply(pred9, function(pd) mean((te9$Apps - drop(pd))^2))
c(M9, lasso_nonzero = sum(as.matrix(coef(cvl9, s = "lambda.min"))[-1, 1] != 0))
round(c(ridge_lambda = cvr9$lambda.min, lasso_lambda = cvl9$lambda.min), 2)
round(test9); round(1 - test9 / mean((te9$Apps - mean(yC[tr9]))^2), 4)

(c) Ridge, at $\lambda'=405.84$, has test error $976{,}261$, and (d) the lasso, at $\lambda'=1.97$, has $1{,}115{,}901$ with all $17$ coefficients nonzero. (g) Least squares has $1{,}135{,}758$, and so have PCR and PLS, whose cross-validation keeps all $17$ components. The test $R^2$ lies between $0.9031$ and ridge's $0.9167$: all five predict `Apps` well, and they differ little. ■

**Exercise 6.10** (ISLR §6.6, exercise 11, with the lasso and ridge regression). Predict the per capita crime rate `crim` of `Boston` with the lasso and ridge regression, evaluated by cross-validation, and compare them with best subset selection and PCR; propose a model, and say whether it involves all the features.

**Solution.** The ten folds of Unit 5's solution of this exercise, now for all four methods, each fold weighted as in Definition 6.29:

In [ ]:
set.seed(1); foldB <- sample(rep(1:10, length = nrow(Boston)))
xB <- model.matrix(crim ~ ., Boston)[, -1]; yB <- Boston$crim; wB <- as.vector(table(foldB)) / nrow(Boston)
cvlB <- cv.glmnet(xB, yB, alpha = 1, foldid = foldB); cvrB <- cv.glmnet(xB, yB, alpha = 0, foldid = foldB)
iminl <- which(cvlB$lambda == cvlB$lambda.min); iminr <- which(cvrB$lambda == cvrB$lambda.min)
mseB <- sapply(1:10, function(j) { fj <- leaps::regsubsets(crim ~ ., data = Boston[foldB != j, ], nvmax = 12)
  mm <- model.matrix(crim ~ ., Boston[foldB == j, ])
  sapply(1:12, function(i) { cf <- coef(fj, id = i); mean((yB[foldB == j] - mm[, names(cf)] %*% cf)^2) }) })
cvB <- drop(mseB %*% wB)
pcrB <- pls::pcr(crim ~ ., data = Boston, scale = TRUE, validation = "CV", segments = split(seq_len(nrow(Boston)), foldB))
cvP <- pls::MSEP(pcrB, estimate = "CV")$val[1, 1, -1]
round(c(lasso = min(cvlB$cvm), ridge = min(cvrB$cvm), best_subset = min(cvB), PCR = min(cvP)), 2)
round(c(lasso_SE = cvlB$cvsd[iminl], ridge_SE = cvrB$cvsd[iminr]), 2)
c(lasso_nonzero = cvlB$nzero[[iminl]], lasso_1se_nonzero = cvlB$nzero[[which(cvlB$lambda == cvlB$lambda.1se)]],
  best_subset_size = which.min(cvB), PCR_components = unname(which.min(cvP)))
cf1 <- as.matrix(coef(cvlB, s = "lambda.1se"))[, 1]; round(cf1[cf1 != 0], 4)

The cross-validation errors are $42.34$ for the lasso, with $11$ nonzero coefficients, $42.35$ for best subset, with $11$ predictors, $42.39$ for PCR, with all $12$ components, which is least squares, and $42.64$ for ridge; the standard errors, $12.98$ for the lasso and $13.33$ for ridge, dwarf the differences. The one-standard-error rule reduces the lasso to `rad` alone, $\widehat{\texttt{crim}}=0.7895+0.2957\,\texttt{rad}$, the predictor that best subset's one-standard-error rule keeps in Unit 5's solution. Proposed model: this lasso fit. It uses one of the twelve features, the others lowering the estimated error by less than one standard error. ■